# DARL DDQN v6: observaciones continuas y barrido AUC/AP–costo

Notebook autónomo. Nueve agentes, semilla 42, 200 camas, 30 días y 1000 episodios por agente.
A prepara XGBoost; B-train entrena DDQN, B-validation selecciona y B-test evalúa.
Las observaciones son continuas; el reward utiliza métricas originales.
Modos: `full`, `smoke`, `plots`, seleccionables con `DARL_DUMMY_MODE`.
Normalización fija: KS y AP identidad, PSI/(1+PSI), (DeltaAUC+1)/2 y validez.
La referencia AUC/AP procede de A-validation; ninguna transformación se ajusta con test.
Base local v5 SHA256: `2260355ff7401ad071071961644bca9e3a8ad447b84730493578850642acfbe3`.

Concepto: un inicio sintético por paciente, ±24 h y probabilidad máxima 10 % de convertir negativos.
Solo nuevos ingresos desde el drift; SepsisLabel permanece intacto.
Versión del inyector: `temporal_random_onset_v1`; metadatos futuros exclusivos del simulador.


## Configuración, reward y observación continua


In [ ]:
# Dependencias estándar para rutas, reproducibilidad, diagnóstico y estructuras de datos.
import os
import json
import time
import random
import hashlib
import platform
import traceback
import itertools
from pathlib import Path
from collections import deque
from dataclasses import dataclass, asdict

# Dependencias para datos, estadística, aprendizaje automático, redes neuronales y gráficos.
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score, precision_recall_curve, auc
from xgboost import XGBClassifier
import xgboost
import sklearn
import torch
from torch import nn
import matplotlib

# Selecciona un backend sin interfaz gráfica para guardar figuras en Kaggle.
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 1. Reproducibilidad
# Reproducibilidad: semilla compartida por los generadores aleatorios del experimento.
SEED = 42

# 2. Configuración del entorno y del experimento
# Modo de ejecución: "smoke" usa una corrida pequeña y "full" el protocolo completo.
RUN_MODE = "full"
MODE = os.environ.get("DARL_DUMMY_MODE", RUN_MODE)

# Permite activar el modo pequeño mediante variable de entorno, con RUN_MODE como valor por defecto.
SMOKE = os.environ.get("DARL_DUMMY_SMOKE", "1" if MODE == "smoke" else "0") == "1"

# Variables fisiológicas usadas como características del modelo.
VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp"]

# Escenarios de deriva evaluados en el experimento.
SCENARIOS = ("stable", "covariate", "concept", "combined")

# Acciones de actualización disponibles para el agente y las políticas de referencia.
ACTIONS = ("A1", "A2", "A3", "A4")

# Camas simuladas en paralelo; se reduce para la corrida smoke.
BEDS = 50 if SMOKE else 200

# Días simulados por replay de B.
DAYS = 8 if SMOKE else 30

# Días usados para preparar el conjunto A.
A_DAYS = 6 if SMOKE else 30

# Días iniciales de A-train dedicados al ajuste temporal de XGBoost.
A_FIT_DAYS = 4 if SMOKE else 20

# Días iniciales de calentamiento antes de habilitar decisiones del agente.
WARMUP = 3

# Días etiquetados que se requieren para actualizar el modelo.
UPDATE_DAYS = 3

# Retraso entre el día observado y la disponibilidad de sus etiquetas.
LABEL_DELAY = 1

# 3. Hiperparámetros de entrenamiento DDQN
# Cantidad mínima de episodios DDQN para cada modo de ejecución.
MIN_EPISODES = 24 if SMOKE else 1000

# Cantidad máxima de episodios DDQN para cada modo de ejecución.
MAX_EPISODES = 24 if SMOKE else 1000

# Tasa de aprendizaje inicial del optimizador DDQN.
LR_START = 5e-4

# Tasa de aprendizaje final prevista durante el decaimiento.
LR_END = 5e-5

# Factor multiplicativo de decaimiento de la exploración epsilon.
EPS_DECAY = 0.99

# Límite inferior de epsilon para conservar exploración durante el entrenamiento.
EPS_MIN = 0.05

# Peso que penaliza el costo de actualización en la recompensa.
COST_LAMBDA = 0.05

# Tolerancia para resolver empates entre valores Q.
Q_TIE = 0.005

# Número de validaciones consecutivas usado al seleccionar el checkpoint.
SELECT_WINDOW = 3

# 4. Parámetros de generación de deriva
# Día de inicio programado para introducir deriva.
ONSET = 5

# Retraso adicional, en días, de la deriva conceptual sintética.
CONCEPT_DELAY = 2

# Nivel máximo de la trayectoria conceptual, usado para normalizar la intensidad temporal al ingreso.
CONCEPT_MAX = 3.0

# Ganancia aplicada al escalamiento de variables bajo deriva.
SCALE_GAIN = 0.4

# Intensidades candidatas de deriva covariable para la calibración.
COV_RATE_GRID = (0.25, 0.35, 0.5)

# Velocidades candidatas de avance por día en la trayectoria conceptual.
CONCEPT_RATE_GRID = (0.1, 0.15, 0.2)

INJECTOR_VERSION = "temporal_random_onset_v1"

# Secuencia de escenarios usada para variar las experiencias de entrenamiento.
TRAIN_CYCLE = (
    "stable",
    "covariate",
    "concept",
    "combined",
    "covariate",
    "concept",
    "combined",
)

# 5. Dispositivo y rutas de salida
# Directorio de resultados: usa DARL_DUMMY_OUTPUT, Kaggle o una ruta local por defecto.
OUT = Path(
    os.environ.get(
        "DARL_DUMMY_OUTPUT",
        (
            "/kaggle/working/temporal_random_onset_v1"
            if Path("/kaggle/input").exists()
            else "kaggle/dqn-dummy/v6/outputs/temporal_random_onset_v1"
        ),
    )
)

# Prepara los subdirectorios para figuras y predicciones exportadas.
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(exist_ok=True)
(OUT / "predictions").mkdir(exist_ok=True)

# Inicializa los generadores aleatorios para obtener resultados reproducibles.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Limita los hilos de Torch para mantener acotado el uso de CPU.
torch.set_num_threads(2)

# Dispositivo disponible para entrenar la red neuronal.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 6. Seguimiento y metadatos de ejecución
# Metadatos acumulados que checkpoint() guarda en run_summary.csv durante la ejecución.
progress = dict(
    version="v6",
    injector_version=INJECTOR_VERSION,
    mode="plots" if MODE == "plots" else ("smoke" if SMOKE else "full"),
    status="running",
    stage="setup",
    seed=SEED,
    days=DAYS,
    beds=BEDS,
    a_days=A_DAYS,
    a_fit_days=A_FIT_DAYS,
    label_delay_days=LABEL_DELAY,
    update_days=UPDATE_DAYS,
    device=str(DEVICE),
    clinical_target="SepsisLabel",
    synthetic_target="scenario_label",
    xgboost_device="cpu",
    python=platform.python_version(),
    numpy=np.__version__,
    pandas=pd.__version__,
    sklearn=sklearn.__version__,
    xgboost=xgboost.__version__,
    torch=torch.__version__,
    # Configuración y progreso del entrenamiento DDQN.
    architecture="15-128-128-4",
    min_episodes=MIN_EPISODES,
    max_episodes=MAX_EPISODES,
    gamma=0.99,  # Factor de descuento registrado para el DDQN.
    batch_size=64,  # Tamaño del minibatch registrado para el DDQN.
    replay_capacity=50000,  # Capacidad del buffer de replay registrada.
    target_copy_updates=50,  # Frecuencia de copia a la red objetivo registrada.
    lr_start=LR_START,
    lr_end=LR_END,
    epsilon_decay=EPS_DECAY,
    epsilon_min=EPS_MIN,
    cost_lambda=COST_LAMBDA,  # Peso del costo de actualización en la recompensa.
    # Parámetros de deriva y umbrales de evaluación.
    concept_max_level=CONCEPT_MAX,
    drift_onset_day=ONSET,
    concept_delay_days=CONCEPT_DELAY,
    observation_transform="KS; PSI/(1+PSI); (DeltaAUC+1)/2; AP; valid",
    validation_every=10,
    selection_window=SELECT_WINDOW,
    kernel_slug="jeffreyamc/darl-dqn-dummy-v6",
)


def checkpoint(stage, **values):
    """Persist progress as CSV after each costly operation."""
    progress.update(stage=stage, **values)
    pd.DataFrame([{"key": k, "value": str(v)} for k, v in progress.items()]).to_csv(
        OUT / "run_summary.csv", index=False
    )
    print(
        f"[{stage}]",
        {
            k: v
            for k, v in values.items()
            if k in ("episodes_done", "status", "selected_episode")
        },
        flush=True,
    )


def table(name, rows):
    """Persist one inspectable table without JSON outputs."""
    data = rows if isinstance(rows, pd.DataFrame) else pd.DataFrame(rows)
    data.to_csv(OUT / f"{name}.csv", index=False)
    return data


@dataclass(frozen=True)
class RewardConfig:
    """Immutable native objective; common evaluation is independent of this choice."""

    alpha: float
    cost_lambda: float

    def __post_init__(self):
        """Reject negative costs and non-convex predictive mixtures."""
        if not 0 <= self.alpha <= 1 or self.cost_lambda < 0:
            raise ValueError("alpha must be in [0,1] and lambda must be nonnegative")

    @property
    def config_id(self):
        """Return a stable CSV/file-safe configuration identifier."""
        return f"a{self.alpha:.2f}_l{self.cost_lambda:.3f}".replace(".", "p")


COMMON_CONFIG = RewardConfig(0.5, 0.025)
CONFIGS = tuple(
    RewardConfig(a, c)
    for a, c in itertools.product((0.0, 0.5, 1.0), (0.01, 0.025, 0.05))
)
OBS_NAMES = ("ks", "psi", "auc_drop", "ap", "valid")
OBS_DIM = 3 * len(OBS_NAMES)


def reset_seed():
    """Restart every independent agent with seed 42 and the same initialization."""
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)


def append_table(name, rows):
    """Append a completed batch, avoiding repeated rewrites of trajectory CSVs."""
    data = rows if isinstance(rows, pd.DataFrame) else pd.DataFrame(rows)
    if len(data):
        path = OUT / f"{name}.csv"
        data.to_csv(path, mode="a", header=not path.exists(), index=False)


def reward_components(metrics, seconds, cost_unit, config):
    """Use raw predictive scores and apply each cost coefficient exactly once."""
    if (
        not np.isfinite(seconds)
        or seconds < 0
        or not np.isfinite(cost_unit)
        or cost_unit <= 0
    ):
        raise ValueError(
            "Time and cost reference must be finite and nonnegative/positive"
        )
    valid = bool(
        metrics["auc_valid"]
        and np.isfinite(metrics["auc"])
        and np.isfinite(metrics["average_precision"])
    )
    auc = float(metrics["auc"]) if valid else 0.0
    ap = float(metrics["average_precision"]) if valid else 0.0
    relative = float(seconds / cost_unit)
    auc_part, ap_part = (1 - config.alpha) * auc, config.alpha * ap
    penalty = config.cost_lambda * relative
    return dict(
        predictive_valid=valid,
        auc_component=auc_part,
        ap_component=ap_part,
        predictive_utility=auc_part + ap_part,
        cost_relative=relative,
        cost_penalty=penalty,
        reward=auc_part + ap_part - penalty,
        common_cost_penalty=COMMON_CONFIG.cost_lambda * relative,
        common_reward=0.5 * auc + 0.5 * ap - COMMON_CONFIG.cost_lambda * relative,
    )


def encode_observation(metrics, last_auc, last_ap, auc_reference, ap_reference):
    """Map five causal observations continuously to [0,1], without fitted scalers."""
    valid = bool(
        metrics.get("auc_valid", False)
        and np.isfinite(metrics["auc"])
        and np.isfinite(metrics["average_precision"])
    )
    auc = (
        metrics["auc"]
        if valid
        else (last_auc if np.isfinite(last_auc) else auc_reference)
    )
    ap = (
        metrics["average_precision"]
        if valid
        else (last_ap if np.isfinite(last_ap) else ap_reference)
    )
    ks, psi = float(metrics["ks"]), float(metrics["psi"])
    if not np.isfinite([ks, psi, auc, ap]).all() or psi < 0:
        raise ValueError("Nonfinite observation or negative PSI")
    result = np.array(
        [ks, psi / (1 + psi), (auc_reference - auc + 1) / 2, ap, float(valid)],
        dtype=np.float32,
    )
    assert result.shape == (5,) and np.isfinite(result).all()
    assert ((result >= -1e-6) & (result <= 1 + 1e-6)).all()
    return result


def select_f1_threshold(labels, probabilities):
    """Maximize positive F1 on A-validation once; choose highest tied threshold."""
    p, r, thresholds = precision_recall_curve(labels, probabilities)
    scores = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-15)
    if len(thresholds) == 0:
        raise ValueError("A-validation cannot identify an F1 threshold")
    return float(thresholds[np.isclose(scores, scores.max(), rtol=0, atol=1e-12)].max())


def classification_metrics(labels, probabilities, threshold=0.5):
    """Compute ranking and fixed-threshold scores with explicit degenerate flags."""
    y, prob = np.asarray(labels, dtype=np.int8), np.asarray(probabilities, dtype=float)
    assert len(y) and len(y) == len(prob) and np.isfinite(prob).all()
    positives, negatives = int((y == 1).sum()), int((y == 0).sum())
    valid = positives > 0 and negatives > 0
    pred = prob >= threshold
    tp, fp = int((pred & (y == 1)).sum()), int((pred & (y == 0)).sum())
    fn, tn = positives - tp, negatives - fp
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, positives)
    fpos = 2 * tp / max(1, 2 * tp + fp + fn)
    fneg = 2 * tn / max(1, 2 * tn + fp + fn)
    if positives:
        pr, re, _ = precision_recall_curve(y, prob)
        ap, area = float(average_precision_score(y, prob)), float(auc(re, pr))
    else:
        ap = area = np.nan
    return dict(
        auc=float(roc_auc_score(y, prob)) if valid else np.nan,
        average_precision=ap,
        pr_auc_trapz=area,
        ap_valid=positives > 0,
        auc_valid=valid,
        f1_positive=fpos,
        f1_macro=(fpos + fneg) / 2,
        precision=precision,
        recall=recall,
        balanced_accuracy=(recall + tn / negatives) / 2 if valid else np.nan,
        threshold=float(threshold),
        prevalence=positives / len(y),
        positives=positives,
        negatives=negatives,
        rows=len(y),
    )


def episode_manifest(episodes):
    """Freeze environment randomization independently from each agent's choices."""
    rng = np.random.default_rng(SEED)
    return pd.DataFrame(
        [
            dict(
                episode=e,
                scenario=TRAIN_CYCLE[(e - 1) % len(TRAIN_CYCLE)],
                replay_id=(e - 1) % 4,
                onset=int(rng.integers(3, 8)) if not SMOKE else ONSET,
                rate_factor=float(rng.choice((0.75, 1.0, 1.25))),
            )
            for e in range(1, episodes + 1)
        ]
    )


RUN_STARTED = time.perf_counter()
RUN_BUDGET_SECONDS = float(os.environ.get("DARL_DUMMY_BUDGET_SECONDS", "39600"))
RUN_OWNED = False
(OUT / "models").mkdir(exist_ok=True)

## Particiones, replay y concept drift temporal por paciente


In [ ]:
def find_parquet():
    """Resolve the mounted Kaggle dataset or the authorized local Parquet."""
    explicit = os.environ.get("DARL_DUMMY_PARQUET")
    paths = (
        [Path(explicit)]
        if explicit
        else sorted(Path("/kaggle/input").rglob("physionet.parquet"))
    )
    paths.append(Path("data/processed/physionet/physionet.parquet"))
    for path in paths:
        if path.is_file():
            return path
    raise FileNotFoundError(
        "physionet.parquet missing; mount jeffreyamc/physionet-sepsis-processed"
    )


def load_frame():
    """Load hourly records and forward fill exclusively within each patient."""
    path = find_parquet()
    frame = pd.read_parquet(
        path, columns=["patient_id", "source_set", "ICULOS", "SepsisLabel", *VITALS]
    )
    assert not frame.duplicated(["patient_id", "ICULOS"]).any()
    assert set(frame.source_set.unique()) == {"a", "b"}
    assert frame.SepsisLabel.isin([0, 1]).all()
    frame = frame.sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)
    assert (
        frame.groupby("patient_id")
        .ICULOS.apply(lambda x: x.is_monotonic_increasing)
        .all()
    )
    frame[VITALS] = frame.groupby("patient_id", sort=False)[VITALS].ffill()
    first_positive = (
        frame.loc[frame.SepsisLabel.eq(1)].groupby("patient_id").ICULOS.min().to_dict()
    )
    patients = (
        frame.groupby(["source_set", "patient_id"], sort=False)
        .SepsisLabel.max()
        .reset_index()
    )
    partitions = {}
    for site in ("a", "b"):
        p = patients.loc[patients.source_set.eq(site)]
        train, rest = train_test_split(
            p.patient_id.to_numpy(),
            test_size=0.4,
            stratify=p.SepsisLabel,
            random_state=SEED,
        )
        val, test = train_test_split(
            rest,
            test_size=0.5,
            stratify=p.set_index("patient_id").loc[rest, "SepsisLabel"],
            random_state=SEED,
        )
        partitions.update(
            {
                f"{site}_train": sorted(train),
                f"{site}_validation": sorted(val),
                f"{site}_test": sorted(test),
            }
        )
    seen = set()
    for ids in partitions.values():
        assert not seen.intersection(ids)
        seen.update(ids)
    assert len(seen) == frame.patient_id.nunique()
    checkpoint("loaded", data_path=str(path), rows=len(frame), patients=len(seen))
    return frame, partitions, first_positive


def make_replay(frame, ids, days=DAYS, beds=BEDS, rng=None):
    """Consume each patient's hourly sequence once; refill a bed next hour."""
    rng = np.random.default_rng(SEED) if rng is None else rng
    assert days >= 1 and beds > 0 and len(set(ids)) == len(ids)
    histories = {
        pid: g.index.to_numpy()
        for pid, g in frame.loc[frame.patient_id.isin(ids)].groupby(
            "patient_id", sort=False
        )
    }
    queue = list(rng.permutation(sorted(ids)))
    if len(queue) < beds:
        raise RuntimeError(
            f"Patient queue exhausted before initialization: {len(queue)} patients, {beds} beds"
        )
    slots = [(queue.pop(), 0) for _ in range(beds)]
    indices = []
    for hour in range(days * 24):
        for bed, (pid, pos) in enumerate(slots):
            indices.append(int(histories[pid][pos]))
            pos += 1
            if pos == len(histories[pid]) and hour < days * 24 - 1:
                if not queue:
                    raise RuntimeError(
                        f"Patient queue exhausted at hour {hour}; requested {days} days, {beds} beds, pool={len(ids)}"
                    )
                slots[bed] = (queue.pop(), 0)
            else:
                slots[bed] = (pid, pos)
    out = frame.loc[indices].reset_index(drop=True).copy()
    out["hour"] = np.repeat(np.arange(days * 24), beds)
    out["bed"] = np.tile(np.arange(beds), days * 24)
    out["day"] = out.hour // 24 + 1
    out["label_available_day"] = out.day + LABEL_DELAY
    assert len(out) == days * 24 * beds and out.groupby("hour").size().eq(beds).all()
    assert not out.duplicated(["patient_id", "ICULOS"]).any()
    assert (
        out.groupby("patient_id")
        .ICULOS.apply(lambda x: x.is_monotonic_increasing)
        .all()
    )
    return out


@dataclass(frozen=True)
class TemporalConceptConfig:
    """Fixed limits for the semisynthetic patient-onset perturbation."""

    seed: int = 42
    max_shift_hours: int = 24
    negative_event_probability: float = 0.10
    version: str = "temporal_random_onset_v1"

    def __post_init__(self):
        """Reject limits outside the agreed protocol, including noninteger hours."""
        if not 0 <= self.max_shift_hours <= 24 or int(self.max_shift_hours) != self.max_shift_hours:
            raise ValueError("max_shift_hours must be an integer in [0, 24]")
        if not 0 <= self.negative_event_probability <= .10:
            raise ValueError("negative_event_probability must be in [0, 0.10]")


def build_patient_metadata(frame, partitions=None):
    """Index COMPLETE histories; simulation metadata must never be model inputs."""
    membership = {pid: name for name, ids in (partitions or {}).items() for pid in ids}
    metadata = {}
    for (site, pid), rows in frame.groupby(["source_set", "patient_id"], sort=False):
        hours = np.sort(rows.ICULOS.to_numpy(dtype=float))
        if len(np.unique(hours)) != len(hours) or not np.isfinite(hours).all():
            raise ValueError(f"Invalid patient history: {site}/{pid}")
        positive = rows.loc[rows.SepsisLabel.eq(1), "ICULOS"]
        metadata[(site, pid)] = dict(
            source_set=site, patient_id=pid, partition=membership.get(pid, str(site)),
            hours=hours, first_iculos=float(hours[0]), last_iculos=float(hours[-1]),
            original_positive=bool(len(positive)),
            original_first_positive=float(positive.min()) if len(positive) else None,
        )
    return metadata


def concept_uniform(config, metadata, admission_hour, purpose):
    """Stable independent draws, shared by policies, reward weights and scenarios."""
    key = json.dumps([str(config.seed), str(metadata["partition"]),
                      str(metadata["source_set"]), str(metadata["patient_id"]),
                      str(int(admission_hour)), purpose], separators=(",", ":"))
    # 53 bits guarantee a double in [0, 1), including at the largest hash value.
    bits = int.from_bytes(hashlib.sha256(key.encode("utf-8")).digest()[:8], "big") >> 11
    return bits / 2**53


def concept_audit_frame(schedule):
    """Recover patient decisions from immutable simulation-only DataFrame attrs."""
    return pd.DataFrame(json.loads(schedule.attrs.get("concept_audit_json", "[]")))


def inject(schedule, scenario, cov_rate, concept_rate, onset, med, iqr,
           patient_metadata=None, concept_config=None):
    """Apply IQR feature drift and one frozen synthetic onset per new admission."""
    if scenario not in SCENARIOS or cov_rate < 0 or concept_rate < 0:
        raise ValueError("Unknown scenario or negative drift rate")
    config = TemporalConceptConfig() if concept_config is None else concept_config
    conceptual = scenario in ("concept", "combined")
    if conceptual and patient_metadata is None:
        raise ValueError("Full-history patient_metadata is required for concept drift")
    out = schedule.copy()
    days = out.day.to_numpy()
    cov = (np.maximum(0, days - onset + 1) * cov_rate
           if scenario in ("covariate", "combined") else np.zeros(len(out)))
    concept_start = int(onset + (CONCEPT_DELAY if scenario == "combined" else 0))
    concept_level = (np.minimum(CONCEPT_MAX, np.maximum(0, days-concept_start+1)*concept_rate)
                     if conceptual else np.zeros(len(out)))
    raw = out[VITALS].to_numpy(dtype=np.float32)
    med_array, iqr_array = np.asarray(med, dtype=np.float32), np.asarray(iqr, dtype=np.float32)
    scale = 1 + SCALE_GAIN * np.minimum(1, cov / 3)
    direction = np.array([1, -1, 1, 1, -1, 1, 1], dtype=np.float32)
    if scenario in ("covariate", "combined"):
        shifted = med_array + scale[:, None]*(raw-med_array) + cov[:, None]*iqr_array*direction
        out[VITALS] = np.where(cov[:, None] > 0, shifted, raw)

    labels = out.SepsisLabel.to_numpy(dtype=np.int8).copy()
    hours = out.ICULOS.to_numpy(dtype=float)
    global_hours = out.hour.to_numpy()
    audit = []
    if conceptual:
        for key, positions in out.groupby(["source_set", "patient_id"], sort=False).indices.items():
            if key not in patient_metadata:
                raise ValueError(f"Missing full history for {key}")
            info = patient_metadata[key]
            admission_hour = int(global_hours[positions].min())
            admission_day = admission_hour // 24 + 1
            intensity = float(np.clip((admission_day-concept_start+1)*concept_rate/CONCEPT_MAX, 0, 1))
            eligible = admission_day >= concept_start and intensity > 0
            maximum = int(np.floor(config.max_shift_hours*intensity))
            probability = config.negative_event_probability*intensity
            original = info["original_first_positive"]
            synthetic = original
            converted = False
            draw = concept_uniform(config, info, admission_hour, "convert")
            shift_draw = concept_uniform(config, info, admission_hour, "shift")
            onset_draw = concept_uniform(config, info, admission_hour, "onset")
            reason = "admitted_before_drift" if admission_day < concept_start else "zero_intensity"
            if eligible:
                if info["original_positive"]:
                    allowed = info["hours"][np.abs(info["hours"]-original) <= maximum]
                    synthetic = float(allowed[int(shift_draw*len(allowed))])
                    reason = "positive_shifted" if synthetic != original else "positive_unchanged"
                elif len(info["hours"]) < 2:
                    reason = "insufficient_history"
                elif draw < probability:
                    allowed = info["hours"][1:]
                    synthetic = float(allowed[int(onset_draw*len(allowed))])
                    converted, reason = True, "negative_converted"
                else:
                    reason = "negative_unchanged"
                labels[positions] = (hours[positions] >= synthetic).astype(np.int8) if synthetic is not None else 0
                ordered = positions[np.argsort(hours[positions])]
                assert np.all(np.diff(labels[ordered]) >= 0)
            visible = bool(np.any(labels[positions] == 1))
            audit.append(dict(
                injector_version=config.version, scenario=scenario, source_set=str(key[0]),
                patient_id=str(key[1]), partition=info["partition"],
                admission_hour=admission_hour, admission_day=admission_day,
                concept_start_day=concept_start, concept_rate=float(concept_rate), intensity=intensity,
                eligible=eligible, max_shift_hours=maximum,
                original_positive=info["original_positive"],
                original_first_positive=original, synthetic_first_positive=synthetic,
                first_iculos=info["first_iculos"], last_iculos=info["last_iculos"],
                effective_shift_hours=synthetic-original if original is not None else None,
                conversion_probability=probability if eligible and not info["original_positive"] else 0.,
                conversion_draw=draw, shift_draw=shift_draw, onset_draw=onset_draw,
                converted=converted, reason=reason, event_visible_in_replay=visible,
                event_censored_in_replay=synthetic is not None and not visible,
                replay_last_iculos=float(hours[positions].max()), replay_rows=len(positions),
            ))
    out["scenario_label"], out["scenario"] = labels, scenario
    out["cov_level"], out["concept_level"] = cov, concept_level
    out.attrs["injector_version"] = config.version
    out.attrs["concept_audit_json"] = json.dumps(audit)
    out.attrs["label_schedule_id"] = hashlib.sha256(
        out.attrs["concept_audit_json"].encode("utf-8")).hexdigest()[:20] if audit else "original_labels"
    assert out.SepsisLabel.equals(schedule.SepsisLabel)
    assert out.scenario_label.isin([0, 1]).all()
    if not conceptual:
        assert np.array_equal(out.scenario_label, out.SepsisLabel.to_numpy())
    return out


def persist_concept_audit(schedule, phase, replay_id):
    """Export each shared label schedule once, independent of reward configuration."""
    audit = concept_audit_frame(schedule)
    if audit.empty:
        return
    payload = schedule.attrs["concept_audit_json"]
    identifier = hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]
    directory = OUT / "concept_audits"
    directory.mkdir(exist_ok=True)
    path = directory / f"{phase}_{replay_id}_{schedule.scenario.iloc[0]}_{identifier}.csv"
    if not path.exists():
        audit.insert(0, "schedule_id", identifier)
        audit.insert(1, "phase", phase)
        audit.insert(2, "replay_id", replay_id)
        audit.to_csv(path, index=False)


TEMPORAL_CONCEPT_CONFIG = TemporalConceptConfig()


## Pipeline selectivo y monitoreo sin discretización


In [ ]:
def build_reference(rows):
    """Freeze raw per-vital deciles and a bounded KS sample."""
    result = {}
    for col in VITALS:
        clean = rows[col].dropna().to_numpy(dtype=float)
        clean = clean[np.isfinite(clean)]
        if not len(clean):
            result[col] = None
            continue
        edges = np.unique(np.quantile(clean, np.linspace(0, 1, 11)))
        edges = (
            np.r_[-np.inf, edges[1:-1], np.inf]
            if len(edges) > 2
            else np.array([-np.inf, np.inf])
        )
        result[col] = dict(
            sample=clean[:: max(1, len(clean) // 1000)][:1000],
            edges=edges,
            hist=np.histogram(clean, bins=edges)[0],
        )
    return result


class Pipeline:
    """Robust preprocessing and XGBoost with independently replaceable stages."""

    def __init__(self, med, iqr, model, ref, params, version=0):
        self.med = np.asarray(med, dtype=np.float32).copy()
        self.iqr = np.asarray(iqr, dtype=np.float32).copy()
        self.model, self.ref, self.params, self.version = (
            model,
            ref,
            dict(params),
            version,
        )

    def transform(self, rows):
        """Impute with past medians, robust-scale and clip values."""
        raw = rows[VITALS].to_numpy(dtype=np.float32)
        return np.clip(
            (np.where(np.isfinite(raw), raw, self.med) - self.med) / self.iqr, -2, 2
        )

    def predict(self, rows):
        """Return probabilities without consulting any labels."""
        return self.model.predict_proba(self.transform(rows))[:, 1]

    def saturation(self, rows):
        """Return the share of feature cells clipped by the current transformer."""
        raw = rows[VITALS].to_numpy(dtype=np.float32)
        return float(
            np.mean(
                np.abs(
                    (np.where(np.isfinite(raw), raw, self.med) - self.med) / self.iqr
                )
                > 2
            )
        )

    def act(self, action, rows):
        """Build a new pipeline using exclusively the supplied labeled past window."""
        assert 0 <= action < 4
        start = time.perf_counter()
        if action == 0:
            return self, time.perf_counter() - start
        med, iqr = self.med, self.iqr
        if action in (1, 3):
            med = (
                rows[VITALS]
                .median()
                .fillna(pd.Series(self.med, index=VITALS))
                .to_numpy(dtype=np.float32)
            )
            q = rows[VITALS].quantile(0.75) - rows[VITALS].quantile(0.25)
            iqr = (
                q.fillna(pd.Series(self.iqr, index=VITALS))
                .clip(lower=1e-6)
                .to_numpy(dtype=np.float32)
            )
        model = self.model
        if action in (2, 3):
            y = rows.scenario_label.to_numpy(dtype=np.int8)
            if np.unique(y).size != 2:
                raise ValueError("Retraining requires both classes in the past window")
            model = XGBClassifier(
                **self.params,
                tree_method="hist",
                device="cpu",
                n_jobs=2,
                random_state=SEED,
                eval_metric="logloss",
                scale_pos_weight=float((y == 0).sum() / max(1, (y == 1).sum())),
            )
            model.fit(
                Pipeline(med, iqr, None, self.ref, self.params).transform(rows), y
            )
        ref = build_reference(rows) if action in (1, 3) else self.ref
        return (
            Pipeline(med, iqr, model, ref, self.params, self.version + 1),
            time.perf_counter() - start,
        )


def fit_pipeline(rows, params):
    """Fit both initial stages from an A-train window only."""
    med = rows[VITALS].median().fillna(0.0).to_numpy(dtype=np.float32)
    iqr = (
        (rows[VITALS].quantile(0.75) - rows[VITALS].quantile(0.25))
        .fillna(1.0)
        .clip(lower=1e-6)
        .to_numpy(dtype=np.float32)
    )
    base = Pipeline(med, iqr, None, build_reference(rows), params)
    fitted, _ = base.act(2, rows)
    fitted.version = 0
    return fitted


def monitor_features(rows, pipe):
    """Compute KS/PSI on exactly the daily rows receiving predictions."""
    ks = []
    psis = []
    for col in VITALS:
        current = rows[col].dropna().to_numpy(dtype=float)
        current = current[np.isfinite(current)]
        ref = pipe.ref[col]
        if not len(current) or ref is None:
            continue
        ks.append(float(ks_2samp(ref["sample"], current, method="asymp").statistic))
        p = ref["hist"].astype(float) + 1e-6
        q = np.histogram(current, bins=ref["edges"])[0].astype(float) + 1e-6
        p /= p.sum()
        q /= q.sum()
        psis.append(float(np.sum((q - p) * np.log(q / p))))
    return dict(
        ks=max(ks, default=0.0),
        psi=max(psis, default=0.0),
        monitored_vitals=len(ks),
        saturation=pipe.saturation(rows),
    )

## Preparación independiente en A y umbral F1 congelado


In [ ]:
def evaluate_initial(pipe, rows, partition, export=False, threshold=0.5):
    """Evaluate fixed predictions on A without fitting on evaluation labels."""
    probs = pipe.predict(rows)
    summary = dict(
        partition=partition,
        **classification_metrics(rows.scenario_label, probs, threshold),
    )
    daily = []
    for d, indices in rows.groupby("day").groups.items():
        daily.append(
            dict(
                partition=partition,
                day=d,
                **classification_metrics(
                    rows.loc[indices, "scenario_label"],
                    probs[np.asarray(indices)],
                    threshold,
                ),
            )
        )
    if export:
        out = rows[
            [
                "patient_id",
                "source_set",
                "ICULOS",
                "hour",
                "day",
                "bed",
                "SepsisLabel",
                "scenario_label",
                "label_available_day",
            ]
        ].copy()
        out["probability"] = probs
        out["pipeline_version"] = 0
        out["active_action"] = "A1"
        out["policy"] = "initial"
        out["scenario"] = "stable"
        out["prediction_hour"] = out.hour
        out["label_available_hour"] = out.day * 24
        out.to_csv(OUT / "predictions/a_test_initial.csv", index=False)
    return summary, daily


def prepare_initial(a_replays):
    """Tune on A-validation; inspect A-test only after choosing the model."""
    train = a_replays["a_train"]
    first = train.loc[train.day.le(A_FIT_DAYS)].reset_index(drop=True)
    tail = train.loc[train.day.gt(A_FIT_DAYS)].reset_index(drop=True)
    results = []
    best = None
    winner = None
    rank = None
    for i, (depth, lr, trees) in enumerate(
        itertools.product((3, 5), (0.05, 0.1), (60, 120))
    ):
        params = dict(max_depth=depth, learning_rate=lr, n_estimators=trees)
        early = fit_pipeline(first, params)
        temporal, _ = evaluate_initial(early, tail, "a_train_future")
        full = fit_pipeline(train, params)
        val, _ = evaluate_initial(full, a_replays["a_validation"], "a_validation")
        if not np.isfinite(val["auc"]):
            raise ValueError(
                "A-validation needs both classes to select hyperparameters"
            )
        row = dict(
            candidate=i,
            **params,
            temporal_auc=temporal["auc"],
            temporal_ap=temporal["average_precision"],
            validation_auc=val["auc"],
            validation_ap=val["average_precision"],
        )
        results.append(row)
        table("xgboost_search", results)
        candidate = (val["auc"], val["average_precision"], -trees, -depth, -lr)
        if rank is None or candidate > rank:
            best, winner, rank = full, row, candidate
    threshold = select_f1_threshold(
        a_replays["a_validation"].scenario_label,
        best.predict(a_replays["a_validation"]),
    )
    table(
        "f1_threshold",
        [
            dict(
                threshold=threshold,
                source="a_validation",
                rule="max_positive_F1_highest_threshold_tie",
            )
        ],
    )
    summary = []
    days = []
    for part in ("a_train", "a_validation", "a_test"):
        info, daily = evaluate_initial(
            best, a_replays[part], part, export=part == "a_test", threshold=threshold
        )
        summary.append(info)
        days.extend(daily)
    table("a_metrics", summary)
    daily = table("a_daily_metrics", days)
    auc_reference = float(daily.loc[daily.partition.eq("a_validation"), "auc"].mean())
    assert np.isfinite(auc_reference)
    checkpoint(
        "initial_model",
        selected_xgboost_candidate=winner["candidate"],
        xgboost_params=str(best.params),
        auc_reference=auc_reference,
        a_test_auc=summary[-1]["auc"],
    )
    ap_reference = float(
        daily.loc[daily.partition.eq("a_validation"), "average_precision"].mean()
    )
    table(
        "observation_reference",
        [
            dict(
                auc_reference=auc_reference,
                ap_reference=ap_reference,
                normalization="fixed_analytic_no_validation_test_fit",
                history=3,
                dimension=OBS_DIM,
            )
        ],
    )
    return best, auc_reference, ap_reference, threshold

## Double DQN con entrada de 15 valores y máscaras


In [ ]:
class QNetwork(nn.Module):
    """Four action values from the last three daily observation blocks."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(OBS_DIM, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 4),
        )

    def forward(self, x):
        """Return unconstrained Q values."""
        return self.net(x)


class ReplayBuffer:
    """FIFO transitions including next-state feasible-action masks."""

    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)
        self.rng = random.Random(SEED)

    def push(self, s, a, r, s2, done, mask):
        """Store immutable transition snapshots."""
        self.buffer.append(
            (s.copy(), int(a), float(r), s2.copy(), bool(done), mask.copy())
        )

    def sample(self, n):
        """Sample uniformly from prior causal decisions."""
        return tuple(np.stack(x) for x in zip(*self.rng.sample(self.buffer, n)))

    def __len__(self):
        """Return the retained transition count."""
        return len(self.buffer)


class DQNAgent:
    """Epsilon-greedy Double DQN; only training episodes update its weights."""

    def __init__(self):
        self.online = QNetwork().to(DEVICE)
        self.target = QNetwork().to(DEVICE)
        self.target.load_state_dict(self.online.state_dict())
        self.target.requires_grad_(False)
        self.optim = torch.optim.Adam(self.online.parameters(), lr=LR_START)
        self.replay = ReplayBuffer()
        self.rng = np.random.default_rng(SEED)
        self.epsilon = 1.0
        self.updates = 0
        self.losses = []

    def q_values(self, s):
        """Return online action values without constructing gradients."""
        with torch.no_grad():
            x = torch.as_tensor(s, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            return self.online(x).squeeze(0).cpu().numpy()

    def select(self, s, mask, greedy=False):
        """Explore or maximize over available maintenance actions."""
        allowed = np.flatnonzero(mask)
        if not greedy and self.rng.random() < self.epsilon:
            return int(self.rng.choice(allowed))
        return int(np.argmax(np.where(mask, self.q_values(s), -np.inf)))

    def learn(self, s, a, r, s2, done, next_mask):
        """Take one masked Double DQN gradient step after a label release."""
        assert np.isfinite(r) and next_mask[0]
        self.replay.push(s, a, r, s2, done, next_mask)
        if len(self.replay) < 64:
            return None
        states, actions, rewards, following, terminals, masks = self.replay.sample(64)
        states = torch.as_tensor(states, dtype=torch.float32, device=DEVICE)
        actions = torch.as_tensor(actions, dtype=torch.long, device=DEVICE).unsqueeze(1)
        following = torch.as_tensor(following, dtype=torch.float32, device=DEVICE)
        rewards = torch.as_tensor(rewards, dtype=torch.float32, device=DEVICE)
        terminals = torch.as_tensor(terminals, dtype=torch.float32, device=DEVICE)
        masks = torch.as_tensor(masks, dtype=torch.bool, device=DEVICE)
        values = self.online(states).gather(1, actions).squeeze(1)
        with torch.no_grad():
            selected = (
                self.online(following)
                .masked_fill(~masks, -torch.inf)
                .argmax(1, keepdim=True)
            )
            next_values = self.target(following).gather(1, selected).squeeze(1)
            targets = rewards + 0.99 * (1 - terminals) * next_values
        loss = nn.functional.smooth_l1_loss(values, targets)
        self.optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_value_(self.online.parameters(), 10.0)
        self.optim.step()
        self.updates += 1
        if self.updates % 50 == 0:
            self.target.load_state_dict(self.online.state_dict())
        self.losses.append(float(loss.item()))
        return self.losses[-1]

    def end_episode(self, fraction):
        """Decay exploration and learning rate once per episode."""
        self.epsilon = max(EPS_MIN, self.epsilon * EPS_DECAY)
        for g in self.optim.param_groups:
            g["lr"] = LR_START + (LR_END - LR_START) * min(1.0, fraction)

    @property
    def lr(self):
        """Return the current learning rate."""
        return self.optim.param_groups[0]["lr"]

    def fingerprint(self):
        """Hash online weights to prove validation/test do not train the agent."""
        h = hashlib.sha256()
        for tensor in self.online.state_dict().values():
            h.update(tensor.detach().cpu().numpy().tobytes())
        return h.hexdigest()

## Entorno causal, métricas, utilidad común y auditoría offline


In [ ]:
class DailyEnv:
    """Persistent pipeline, daily frozen predictions and one-day delayed labels."""

    def __init__(
        self,
        schedule,
        initial,
        auc_reference,
        cost_unit,
        ap_reference=0.0,
        reward_config=COMMON_CONFIG,
        threshold=0.5,
        prediction_path=None,
        policy="dqn",
        phase="train",
        episode=0,
    ):
        self.schedule = schedule
        self.parts = {
            int(d): r.reset_index(drop=True)
            for d, r in schedule.groupby("day", sort=True)
        }
        self.days = int(schedule.day.max())
        assert self.days > WARMUP
        self.pipe, self.auc_reference, self.ap_reference = (
            initial,
            auc_reference,
            ap_reference,
        )
        self.last_auc, self.last_ap = auc_reference, ap_reference
        self.cost_unit, self.config, self.threshold = (
            cost_unit,
            reward_config,
            threshold,
        )
        self.prediction_path, self.policy = prediction_path, policy
        self.context = dict(
            injector_version=schedule.attrs.get("injector_version", INJECTOR_VERSION),
            label_schedule_id=schedule.attrs.get("label_schedule_id", "original_labels"),
            config_id=reward_config.config_id,
            alpha=reward_config.alpha,
            cost_lambda=reward_config.cost_lambda,
            phase=phase,
            episode=episode,
        )
        self.history, self.daily, self.actions = deque(maxlen=3), [], []
        self.pending_probabilities = None
        self.day = 1
        for d in range(1, WARMUP + 1):
            self.day = d
            pending = self._predict(d, 0, 0.0, None)
            self.day = d + 1
            self._release(pending, active=False)

    def state(self):
        """Concatenate only already released blocks, oldest to newest."""
        assert len(self.history) == 3 and self.daily[-1]["day"] == self.day - 1
        state = np.concatenate(self.history).astype(np.float32)
        assert state.shape == (OBS_DIM,) and np.isfinite(state).all()
        return state

    def training_rows(self):
        """Use only the three preceding days with available labels."""
        out = pd.concat(
            [
                self.parts[d]
                for d in range(
                    max(1, self.day - UPDATE_DAYS), min(self.day, self.days + 1)
                )
            ],
            ignore_index=True,
        )
        assert out.day.max() < self.day and out.label_available_day.max() <= self.day
        return out

    def action_mask(self):
        """Prevent A3/A4 classifier fits on a single-class update window."""
        past = self.training_rows()
        both = past.scenario_label.nunique() == 2
        return np.array([True, len(past) > 0, both, both], dtype=bool)

    def _predict(self, day, action, elapsed, state):
        rows = self.parts[day]
        prob = np.array(self.pipe.predict(rows), copy=True)
        self.pending_probabilities = prob.copy()
        return dict(
            day=day,
            rows=rows,
            probability=prob,
            metrics=monitor_features(rows, self.pipe),
            action=action,
            elapsed=float(elapsed),
            version=self.pipe.version,
            state=state,
        )

    def _release(self, pending, active):
        day, rows, prob = pending["day"], pending["rows"], pending["probability"]
        assert (
            self.day == day + LABEL_DELAY
            and rows.label_available_day.le(self.day).all()
        )
        observed = classification_metrics(rows.scenario_label, prob, self.threshold)
        original = (
            observed
            if np.array_equal(rows.scenario_label, rows.SepsisLabel)
            else classification_metrics(rows.SepsisLabel, prob, self.threshold)
        )
        components = reward_components(
            observed, pending["elapsed"], self.cost_unit, self.config
        )
        record = dict(
            **self.context,
            day=day,
            policy=self.policy,
            scenario=rows.scenario.iloc[0],
            **pending["metrics"],
            **observed,
            **{f"{k}_original": v for k, v in original.items()},
            auc_drop=self.auc_reference - observed["auc"],
            action=ACTIONS[pending["action"]],
            **components,
            time_s=pending["elapsed"],
            cost_unit_s=self.cost_unit,
            pipeline_version=pending["version"],
            label_received_day=self.day,
            warmup=not active,
            patients=int(rows.patient_id.nunique()),
            cov_level=float(rows.cov_level.iloc[0]),
            concept_level=float(rows.concept_level.iloc[0]),
        )
        block = encode_observation(
            record, self.last_auc, self.last_ap, self.auc_reference, self.ap_reference
        )
        record.update(
            {f"obs_{name}": float(block[i]) for i, name in enumerate(OBS_NAMES)}
        )
        if not active:
            record["reward"] = record["common_reward"] = np.nan
        self.daily.append(record)
        self.history.append(block)
        if components["predictive_valid"]:
            self.last_auc, self.last_ap = observed["auc"], observed["average_precision"]
        if self.prediction_path is not None:
            cols = [
                "patient_id",
                "source_set",
                "ICULOS",
                "hour",
                "day",
                "bed",
                "SepsisLabel",
                "scenario_label",
                "label_available_day",
            ]
            out = rows[cols].copy()
            out["probability"], out["pipeline_version"], out["active_action"] = (
                prob,
                pending["version"],
                record["action"],
            )
            out["prediction_hour"], out["label_available_hour"] = out.hour, out.day * 24
            out["policy"], out["scenario"], out["config_id"] = (
                self.policy,
                record["scenario"],
                self.config.config_id,
            )
            path = Path(self.prediction_path)
            out.to_csv(path, mode="a", header=not path.exists(), index=False)
        return record

    def step(self, requested):
        """Choose with past labels, predict, and close reward on the next day."""
        if self.day > self.days:
            raise RuntimeError("Episode already finished")
        d, state, past, mask = (
            self.day,
            self.state(),
            self.training_rows(),
            self.action_mask(),
        )
        executed = int(requested) if mask[int(requested)] else 0
        self.pipe, seconds = self.pipe.act(executed, past)
        pending = self._predict(d, executed, seconds, state.copy())
        self.day = d + LABEL_DELAY
        result = self._release(pending, active=True)
        info = dict(
            result,
            requested=int(requested),
            executed=executed,
            state_end_day=d - 1,
            training_min_day=int(past.day.min()),
            training_max_day=int(past.day.max()),
            training_label_available_max=int(past.label_available_day.max()),
            fallback_reason="" if requested == executed else "one_class_past_window",
            **{f"mask_{a}": bool(mask[i]) for i, a in enumerate(ACTIONS)},
            **{f"state_{i}": float(v) for i, v in enumerate(state)},
        )
        self.actions.append(info)
        return self.state(), result["reward"], d == self.days, info


def summarize_roll(daily, actions, threshold, probabilities=None, labels=None):
    """Keep 30-day means, 27-decision means and pooled metrics distinct."""
    first = actions.iloc[0]
    result = {
        k: first[k]
        for k in [
            "config_id",
            "alpha",
            "cost_lambda",
            "phase",
            "episode",
            "scenario",
            "policy",
        ]
    }
    result.update(
        native_return=float(actions.reward.sum()),
        common_return=float(actions.common_reward.sum()),
        cost_relative=float(actions.cost_relative.sum()),
        cost_penalty=float(actions.cost_penalty.sum()),
        seconds=float(actions.time_s.sum()),
        decisions=len(actions),
        days=len(daily),
        mean_daily_ap=float(daily.average_precision.mean()),
        mean_auc=float(daily.auc.mean()),
        decision_mean_auc=float(actions.auc.mean()),
        decision_mean_ap=float(actions.average_precision.mean()),
        valid_days=int(daily.auc_valid.sum()),
        **{
            f"count_{a}": int(actions.executed.eq(i).sum())
            for i, a in enumerate(ACTIONS)
        },
    )
    for name in [
        "pr_auc_trapz",
        "f1_positive",
        "f1_macro",
        "precision",
        "recall",
        "balanced_accuracy",
        "auc_original",
        "average_precision_original",
        "prevalence",
    ]:
        result[f"mean_{name}"] = float(daily[name].mean())
    if probabilities is not None:
        result.update(
            {
                f"pooled_{k}": v
                for k, v in classification_metrics(
                    labels, probabilities, threshold
                ).items()
            }
        )
    return result


def audit_alternatives(
    before, past, today, mask, chosen, info, config, threshold, cost_unit
):
    """Audit after selection; reuse actual chosen outcome and its measured time."""
    entries = []
    for a in range(4):
        if not mask[a]:
            continue
        if a == chosen:
            metrics = {k: info[k] for k in ["auc", "average_precision", "auc_valid"]}
            seconds = info["time_s"]
        else:
            changed, seconds = before.act(a, past)
            metrics = classification_metrics(
                today.scenario_label, changed.predict(today), threshold
            )
        entry = {
            k: info[k]
            for k in [
                "config_id",
                "alpha",
                "cost_lambda",
                "phase",
                "episode",
                "scenario",
                "day",
            ]
        }
        entry.update(
            action=ACTIONS[a],
            executed=a,
            chosen=a == chosen,
            auc=metrics["auc"],
            average_precision=metrics["average_precision"],
            time_s=seconds,
            **reward_components(metrics, seconds, cost_unit, config),
        )
        entries.append(entry)
    best = max(e["common_reward"] for e in entries)
    best_native = max(e["reward"] for e in entries)
    for e in entries:
        e.update(
            oracle_common_utility=best,
            chosen_common_utility=info["common_reward"],
            regret=best - info["common_reward"],
            native_regret=best_native - info["reward"],
            oracle_best=bool(np.isclose(e["common_reward"], best, rtol=0, atol=1e-12)),
        )
    assert all(e["regret"] >= -1e-12 and e["native_regret"] >= -1e-12 for e in entries)
    return entries


def choose_policy(policy, agent, state, mask, metrics, rng, auc_reference):
    """Use raw diagnostic cutoffs only in the inherited heuristic, not the state."""
    if policy == "dqn":
        return agent.select(state, mask, greedy=True)
    if policy in ACTIONS:
        return ACTIONS.index(policy)
    if policy == "random":
        return int(rng.choice(np.flatnonzero(mask)))
    cov = metrics["ks"] >= 0.30 or metrics["psi"] >= 0.25
    low = np.isfinite(metrics["auc"]) and auc_reference - metrics["auc"] >= 0.03
    requested = 3 if cov and low else 1 if cov else 2 if low else 0
    return requested if mask[requested] else 0


def roll(
    schedule,
    policy,
    initial,
    auc_reference,
    cost_unit,
    agent=None,
    audit=False,
    export=False,
    ap_reference=0.0,
    reward_config=COMMON_CONFIG,
    threshold=0.5,
    phase="validation",
    episode=0,
):
    """Evaluate a frozen controller; retain outcomes, causal states and optional audits."""
    tag = reward_config.config_id if policy == "dqn" else "baseline"
    path = (
        (OUT / "predictions" / f"b_test_{tag}_{schedule.scenario.iloc[0]}_{policy}.csv")
        if export
        else None
    )
    if path is not None and path.exists():
        raise FileExistsError(f"Refusing mixed prediction runs: {path}")
    before_hash = agent.fingerprint() if agent is not None else None
    env = DailyEnv(
        schedule,
        initial,
        auc_reference,
        cost_unit,
        ap_reference,
        reward_config,
        threshold,
        path,
        policy,
        phase,
        episode,
    )
    rng = np.random.default_rng(SEED)
    audits = []
    while env.day <= env.days:
        state, mask = env.state(), env.action_mask()
        action = choose_policy(
            policy, agent, state, mask, env.daily[-1], rng, auc_reference
        )
        q = (
            agent.q_values(state)
            if agent is not None and policy == "dqn"
            else np.full(4, np.nan)
        )
        before, past, today = env.pipe, env.training_rows(), env.parts[env.day]
        _, _, done, info = env.step(action)
        info.update({f"q_{a}": float(q[i]) for i, a in enumerate(ACTIONS)})
        if audit:
            audits.extend(
                audit_alternatives(
                    before,
                    past,
                    today,
                    mask,
                    info["executed"],
                    info,
                    reward_config,
                    threshold,
                    cost_unit,
                )
            )
        if done:
            break
    if agent is not None:
        assert agent.fingerprint() == before_hash
    daily, actions = pd.DataFrame(env.daily), pd.DataFrame(env.actions)
    # Read prediction files only for pooled test metrics, never for controller decisions.
    saved = (
        pd.read_csv(path, usecols=["SepsisLabel", "scenario_label", "probability"])
        if path is not None
        else None
    )
    result = summarize_roll(
        daily,
        actions,
        threshold,
        saved.probability.to_numpy() if saved is not None else None,
        saved.scenario_label.to_numpy() if saved is not None else None,
    )
    if saved is not None:
        result.update(
            {
                f"pooled_{k}_original": v
                for k, v in classification_metrics(
                    saved.SepsisLabel, saved.probability, threshold
                ).items()
            }
        )
    assert len(actions) == int(schedule.day.max()) - WARMUP
    return result, actions, daily, pd.DataFrame(audits)

## Calibración de deriva y costo compartida en B-train


In [ ]:
def calibrate(base, initial, auc_reference, patient_metadata):
    """Choose slow, measurable stress using B-train exclusively; report unmet gates."""
    stable = inject(base, "stable", 0, 0, ONSET, initial.med, initial.iqr)
    history = stable.loc[stable.day.between(1, 3)]
    if history.scenario_label.nunique() != 2:
        raise ValueError("B-train cost window lacks both classes")
    timings = [initial.act(3, history)[1] for _ in range(5)]
    cost_unit = max(float(np.median(timings)), 1e-6)
    table(
        "cost_calibration",
        [
            dict(repetition=i + 1, a4_seconds=v, cost_unit=cost_unit)
            for i, v in enumerate(timings)
        ],
    )
    candidates = []
    rates = {}
    for kind, grid, repair in (
        ("covariate", COV_RATE_GRID, 1),
        ("concept", CONCEPT_RATE_GRID, 2),
    ):
        for rate in grid:
            scenario = inject(
                base,
                kind,
                rate if kind == "covariate" else 0.0,
                rate if kind == "concept" else 0.0,
                ONSET,
                initial.med,
                initial.iqr,
                patient_metadata,
                TEMPORAL_CONCEPT_CONFIG,
            )
            persist_concept_audit(scenario, "calibration", f"{kind}_{rate:g}")
            first = min(ONSET, DAYS)
            last = DAYS

            def value(rows, pipe):
                return classification_metrics(rows.scenario_label, pipe.predict(rows))[
                    "auc"
                ]

            early0 = value(stable.loc[stable.day.eq(first)], initial)
            late0 = value(stable.loc[stable.day.eq(last)], initial)
            early = value(scenario.loc[scenario.day.eq(first)], initial)
            late = value(scenario.loc[scenario.day.eq(last)], initial)
            past = scenario.loc[scenario.day.between(last - 3, last - 1)]
            changed, _ = (
                initial.act(repair, past)
                if repair == 1 or past.scenario_label.nunique() == 2
                else (initial, 0.0)
            )
            recovered = value(scenario.loc[scenario.day.eq(last)], changed)
            qualified = bool(
                np.isfinite([early0, late0, early, late, recovered]).all()
                and early0 - early <= 0.05
                and late0 - late >= 0.10
                and recovered - late >= 0.05
            )
            candidates.append(
                dict(
                    kind=kind,
                    rate=rate,
                    early_drop=early0 - early,
                    late_drop=late0 - late,
                    recovery=recovered - late,
                    qualified=qualified,
                )
            )
        subset = pd.DataFrame([r for r in candidates if r["kind"] == kind])
        qualified = subset.loc[subset.qualified]
        chosen = (
            qualified.sort_values("rate")
            if len(qualified)
            else subset.sort_values(
                ["late_drop", "rate"], ascending=[False, True], na_position="last"
            )
        ).iloc[0]
        rates[kind] = float(chosen.rate)
    report = table("calibration", candidates)
    calibrated = bool(report.groupby("kind").qualified.any().all())
    checkpoint(
        "calibrated",
        cov_rate=rates["covariate"],
        concept_rate_level_per_day=rates["concept"],
        cost_unit_s=cost_unit,
        calibration_qualified=calibrated,
    )
    return rates["covariate"], rates["concept"], cost_unit, calibrated

## Barrido controlado de nueve agentes y selección común


In [ ]:
def history_predictive_fields(result):
    """Keep episode-wide and decision-day means distinct in training exports."""
    return {
        key: result[key]
        for key in (
            "mean_auc",
            "mean_daily_ap",
            "decision_mean_auc",
            "decision_mean_ap",
        )
    }


def masked_tie_agreement(reference_q, candidate_q, masks, tolerance=Q_TIE):
    """Compare feasible choices on identical states, allowing small Q ties."""
    ref = np.argmax(np.where(masks, reference_q, -np.inf), axis=1)
    other = np.argmax(np.where(masks, candidate_q, -np.inf), axis=1)
    idx = np.arange(len(ref))
    delta = np.abs(reference_q[idx, ref] - reference_q[idx, other])
    return float(np.mean((ref == other) | (delta <= tolerance)))


def select_checkpoint_index(checkpoints):
    """Rank complete three-checkpoint common-utility windows without consulting test."""
    if len(checkpoints) < SELECT_WINDOW:
        raise ValueError("Three complete validation checkpoints are required")
    candidates = []
    for i in range(SELECT_WINDOW - 1, len(checkpoints)):
        window = checkpoints[i - SELECT_WINDOW + 1 : i + 1]
        mean = lambda key: float(np.mean([c[key] for c in window]))
        candidates.append(
            (
                mean("common_score"),
                mean("decision_mean_ap"),
                mean("decision_mean_auc"),
                -mean("seconds"),
                -i,
                i,
            )
        )
    return max(candidates)[-1]


def train_agent(
    train_bases,
    val_base,
    initial,
    auc_reference,
    cost_unit,
    cov_rate,
    concept_rate,
    patient_metadata,
    ap_reference,
    threshold,
    config,
    manifest,
):
    """Train all requested episodes, checkpointing by a shared validation objective."""
    reset_seed()
    agent = DQNAgent()
    history, checkpoints = [], []
    pending_daily, pending_actions = [], []
    vals = [
        inject(
            val_base,
            s,
            cov_rate,
            concept_rate,
            ONSET,
            initial.med,
            initial.iqr,
            patient_metadata,
            TEMPORAL_CONCEPT_CONFIG,
        )
        for s in SCENARIOS
    ]
    for schedule in vals:
        persist_concept_audit(schedule, "validation", "b_validation")
    context = dict(
        config_id=config.config_id, alpha=config.alpha, cost_lambda=config.cost_lambda
    )
    common_args = dict(
        ap_reference=ap_reference, reward_config=config, threshold=threshold
    )
    started = time.perf_counter()
    for spec in manifest.itertuples(index=False):
        ep = int(spec.episode)
        schedule = inject(
            train_bases[int(spec.replay_id)],
            spec.scenario,
            cov_rate * spec.rate_factor,
            concept_rate * spec.rate_factor,
            int(spec.onset),
            initial.med,
            initial.iqr,
            patient_metadata,
            TEMPORAL_CONCEPT_CONFIG,
        )
        persist_concept_audit(schedule, "train", int(spec.replay_id))
        env = DailyEnv(
            schedule,
            initial,
            auc_reference,
            cost_unit,
            phase="train",
            episode=ep,
            **common_args,
        )
        epsilon, lr = agent.epsilon, agent.lr
        loss_start = len(agent.losses)
        while env.day <= env.days:
            state = env.state()
            action = agent.select(state, env.action_mask())
            state2, reward, done, info = env.step(action)
            agent.learn(state, action, reward, state2, done, env.action_mask())
            if done:
                break
        actions = pd.DataFrame(env.actions)
        daily = pd.DataFrame(env.daily)
        result = summarize_roll(daily, actions, threshold)
        recent_losses = agent.losses[loss_start:]
        row = dict(
            **context,
            phase="train",
            episode=ep,
            scenario=spec.scenario,
            injector_version=schedule.attrs["injector_version"],
            label_schedule_id=schedule.attrs["label_schedule_id"],
            replay_id=int(spec.replay_id),
            onset=int(spec.onset),
            rate_factor=spec.rate_factor,
            train_return=result["native_return"],
            train_common_return=result["common_return"],
            **history_predictive_fields(result),
            seconds=result["seconds"],
            cost_relative=result["cost_relative"],
            epsilon=epsilon,
            lr=lr,
            updates=agent.updates,
            loss=float(np.mean(recent_losses)) if recent_losses else np.nan,
            last_loss=agent.losses[-1] if agent.losses else np.nan,
            validation_return=np.nan,
            validation_common_return=np.nan,
            agreement_prev=np.nan,
            **{f"train_count_{a}": result[f"count_{a}"] for a in ACTIONS},
            **{f"val_count_{a}": np.nan for a in ACTIONS},
        )
        pending_daily.extend(env.daily)
        pending_actions.extend(env.actions)
        agent.end_episode(ep / MAX_EPISODES)
        if ep % 10 == 0 or ep == MAX_EPISODES:
            scores, signature, q, states, masks = [], [], [], [], []
            counts = np.zeros(4, dtype=int)
            for val in vals:
                out, ac, dy, _ = roll(
                    val,
                    "dqn",
                    initial,
                    auc_reference,
                    cost_unit,
                    agent,
                    phase="validation",
                    episode=ep,
                    **common_args,
                )
                scores.append(out)
                append_table("validation_history", [out])
                append_table("validation_daily_metrics", dy)
                append_table("validation_actions", ac)
                signature.extend(ac.executed.to_list())
                q.append(ac[[f"q_{a}" for a in ACTIONS]].to_numpy())
                states.append(
                    ac[[f"state_{i}" for i in range(OBS_DIM)]].to_numpy(
                        dtype=np.float32
                    )
                )
                masks.append(ac[[f"mask_{a}" for a in ACTIONS]].to_numpy(dtype=bool))
                counts += np.bincount(ac.executed, minlength=4)
            row["validation_return"] = float(
                np.mean([r["native_return"] for r in scores])
            )
            row["validation_common_return"] = float(
                np.mean([r["common_return"] for r in scores])
            )
            current = dict(
                episode=ep,
                common_score=row["validation_common_return"],
                native_score=row["validation_return"],
                decision_mean_ap=float(
                    np.mean([r["decision_mean_ap"] for r in scores])
                ),
                decision_mean_auc=float(
                    np.mean([r["decision_mean_auc"] for r in scores])
                ),
                seconds=float(np.mean([r["seconds"] for r in scores])),
                signature=np.asarray(signature),
                q=np.concatenate(q),
                probe_states=np.concatenate(states),
                probe_masks=np.concatenate(masks),
                weights={
                    k: v.detach().cpu().clone()
                    for k, v in agent.online.state_dict().items()
                },
            )
            if checkpoints:
                row["agreement_prev"] = float(
                    np.mean(current["signature"] == checkpoints[-1]["signature"])
                )
            row.update(
                {f"val_count_{a}": int(counts[i]) for i, a in enumerate(ACTIONS)}
            )
            checkpoints.append(current)
            append_table("train_daily_metrics", pending_daily)
            append_table("train_actions", pending_actions)
            pending_daily.clear()
            pending_actions.clear()
            # Persist each config's small history separately, then consolidate at completion.
            table(f"history_{config.config_id}", history + [row])
            append_table(
                "controller_freeze",
                [
                    dict(
                        **context,
                        phase="validation",
                        episode=ep,
                        weights_sha256=agent.fingerprint(),
                        passed=True,
                        oracle_training_transitions=0,
                    )
                ],
            )
            checkpoint(
                "training",
                active_config=config.config_id,
                episodes_done=ep,
                total_configs=len(CONFIGS),
                config_elapsed_s=time.perf_counter() - started,
                validation_common_return=row["validation_common_return"],
            )
            if time.perf_counter() - RUN_STARTED > RUN_BUDGET_SECONDS:
                raise TimeoutError(
                    "Budget guard: partial outputs saved; requested episodes were not reduced"
                )
        history.append(row)
    assert len(history) == MAX_EPISODES
    index = select_checkpoint_index(checkpoints)
    winner = checkpoints[index]
    agent.online.load_state_dict(winner["weights"])
    agent.target.load_state_dict(winner["weights"])
    selected = checkpoints[index - SELECT_WINDOW + 1 : index + 1]
    scores = np.array([c["common_score"] for c in selected])
    probe = QNetwork().to(DEVICE)
    probe_states = torch.as_tensor(winner["probe_states"], device=DEVICE)
    ties = []
    for c in selected[:-1]:
        probe.load_state_dict(c["weights"])
        with torch.no_grad():
            candidate = probe(probe_states).cpu().numpy()
        ties.append(masked_tie_agreement(winner["q"], candidate, winner["probe_masks"]))
    summary = dict(
        **context,
        selected_episode=winner["episode"],
        episodes=len(history),
        updates=agent.updates,
        selected_common_return=winner["common_score"],
        selected_native_return=winner["native_score"],
        selection_score=float(scores.mean()),
        selection_ap=float(np.mean([c["decision_mean_ap"] for c in selected])),
        selection_auc=float(np.mean([c["decision_mean_auc"] for c in selected])),
        selection_seconds=float(np.mean([c["seconds"] for c in selected])),
        selected_window_variation=float(np.ptp(scores) / max(abs(scores.mean()), 1e-6)),
        selected_window_agreement=min(
            float(np.mean(c["signature"] == winner["signature"])) for c in selected[:-1]
        ),
        selected_window_tie_agreement=min(ties),
        convergence_assessable=True,
        elapsed_s=time.perf_counter() - started,
        weights_sha256=agent.fingerprint(),
    )
    append_table(
        "checkpoint_history",
        [
            dict(
                **context,
                episode=c["episode"],
                common_score=c["common_score"],
                native_score=c["native_score"],
                decision_mean_ap=c["decision_mean_ap"],
                decision_mean_auc=c["decision_mean_auc"],
                seconds=c["seconds"],
                selected=c is winner,
            )
            for c in checkpoints
        ],
    )
    append_table("checkpoint_selection", [summary])
    append_table("training_history", history)
    torch.save(winner["weights"], OUT / "models" / f"{config.config_id}.pt")
    return summary

## Figuras regenerables exclusivamente desde CSV


In [ ]:
def render_temporal_concept_from_csv(root):
    """Render patient timelines and aggregate diagnostics exclusively from CSV."""
    root = Path(root)
    paths = [root / name for name in ("concept_patients.csv", "concept_timelines.csv", "drift_calendar.csv")]
    if not any(path.exists() for path in paths[:2]):
        return []  # Historical outputs from the old injector remain readable.
    if not all(path.exists() for path in paths):
        raise FileNotFoundError("Temporal concept plotting requires patients, timelines and calendar CSV")
    patients = pd.read_csv(paths[0], dtype={"patient_id": str, "source_set": str})
    timelines = pd.read_csv(paths[1], dtype={"patient_id": str, "source_set": str})
    calendar = pd.read_csv(paths[2])
    destination = root / "figures"
    destination.mkdir(exist_ok=True)
    manifest, summary = [], []

    def save(fig, name):
        fig.tight_layout(rect=(0, 0, 1, .95))
        path = destination / f"{name}.png"
        fig.savefig(path, dpi=140, bbox_inches="tight")
        plt.close(fig)
        manifest.append(dict(family="temporal_concept", config_id="shared", path=path.relative_to(root).as_posix()))

    for scenario, cohort in patients.groupby("scenario", sort=False):
        traces = timelines.loc[timelines.scenario.eq(scenario)]
        groups = list(traces.groupby(["source_set", "patient_id"], sort=False))
        if groups:
            fig, axes = plt.subplots(len(groups), 1, figsize=(12, 2.1*len(groups)), squeeze=False)
            for ax, ((site, pid), rows) in zip(axes.flat, groups):
                rows = rows.sort_values("ICULOS")
                ax.step(rows.ICULOS, rows.SepsisLabel, where="post", ls="--", lw=2.4,
                        color="#327ea4", label="SepsisLabel original")
                ax.step(rows.ICULOS, rows.scenario_label, where="post", lw=1.6,
                        color="#d58426", label="scenario_label sintético")
                ax.fill_between(rows.ICULOS, rows.SepsisLabel, rows.scenario_label,
                                step="post", color="#d58426", alpha=.15)
                first = rows.iloc[0]
                drift_hour = rows.ICULOS.iloc[0] + (first.concept_start_day-1)*24 - rows.hour.iloc[0]
                if rows.ICULOS.min() <= drift_hour <= rows.ICULOS.max():
                    ax.axvline(drift_hour, color="gray", ls=":", label="Inicio global del drift")
                reasons = {"admitted_before_drift": "Ingreso anterior: sin cambios",
                           "positive_shifted": "Inicio positivo desplazado",
                           "positive_unchanged": "Inicio positivo conservado",
                           "negative_converted": "Nuevo positivo sintético",
                           "negative_unchanged": "Permanece negativo",
                           "zero_intensity": "Intensidad cero",
                           "insufficient_history": "Historia insuficiente"}
                original = f"{first.original_first_positive:g} h" if pd.notna(first.original_first_positive) else "sin evento"
                synthetic = f"{first.synthetic_first_positive:g} h" if pd.notna(first.synthetic_first_positive) else "sin evento"
                ax.set(title=f"{site}/{pid} · ingreso día {first.admission_day} · "
                             f"{reasons.get(first.reason, first.reason)} · {original} → {synthetic}",
                       ylabel="Etiqueta", ylim=(-.1, 1.1), yticks=[0, 1])
                ax.grid(alpha=.2)
            axes[0, 0].legend(loc="upper left", bbox_to_anchor=(0, 1.52), ncol=3, fontsize=8)
            axes[-1, 0].set_xlabel("Hora en UCI del paciente; no día global del hospital")
            fig.suptitle(f"Inicio temporal por paciente · {SCENARIO_LABELS.get(scenario, scenario)}")
            save(fig, f"13_concept_trayectorias_{scenario}")

        positive = cohort.loc[cohort.original_positive & cohort.eligible]
        fig, ax = plt.subplots(figsize=(9, 4))
        edges = np.arange(-24.5, 25.5, 1)
        counts, edges, _ = ax.hist(positive.effective_shift_hours, bins=edges, color="#327ea4")
        ax.axvline(0, color="gray", ls="--")
        ax.set(title=f"Adelantos y retrasos de positivos · {scenario}",
               xlabel="Desplazamiento efectivo (horas); negativo = adelanto", ylabel="Pacientes")
        save(fig, f"14_concept_desplazamientos_{scenario}")
        for left, right, count in zip(edges[:-1], edges[1:], counts):
            summary.append(dict(scenario=scenario, kind="shift_histogram", category="positive_eligible",
                                left=left, right=right, count=int(count)))

        daily = calendar.loc[calendar.scenario.eq(scenario)].sort_values("day")
        # Missing days in small synthetic checks are not interpolated.
        daily = daily.set_index("day").reindex(range(1, int(daily.day.max())+1)).rename_axis("day").reset_index()
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(daily.day, daily.original_prevalence, "--", label="SepsisLabel original", color="#327ea4")
        ax.plot(daily.day, daily.scenario_prevalence, label="scenario_label sintético", color="#d58426")
        ax.axvline(cohort.concept_start_day.iloc[0], color="gray", ls=":", label="Inicio conceptual")
        ax.set(title=f"Prevalencia diaria por registro · {scenario}", xlabel="Día simulado", ylabel="Fracción positiva")
        ax.legend()
        save(fig, f"15_concept_prevalencia_{scenario}")

        # Disjoint patient groups; original positives include shifted and unmodified events.
        values = [int(cohort.original_positive.sum()), int(cohort.converted.sum()),
                  int((~cohort.original_positive & ~cohort.converted).sum())]
        labels = ["Positivos originales\n(evento conservado)", "Negativos\nconvertidos", "Negativos\nsin cambios"]
        fig, ax = plt.subplots(figsize=(10, 4))
        bars = ax.bar(labels, values, color=["#327ea4", "#d58426", "#8b9fa9"])
        ax.bar_label(bars, padding=3)
        ax.margins(y=.15)
        ax.set(title=f"Resultado por paciente · {scenario}", ylabel="Pacientes")
        save(fig, f"16_concept_conteos_{scenario}")
        for label, count in zip(labels, values):
            summary.append(dict(scenario=scenario, kind="patient_count", category=label.replace("\n", " "), count=count))
    pd.DataFrame(summary).to_csv(root / "concept_plot_summary.csv", index=False)
    pd.DataFrame(manifest).to_csv(root / "concept_figure_manifest.csv", index=False)
    return manifest


ACTION_COLORS = {"A1": "#8b9fa9", "A2": "#4796ab", "A3": "#ca9145", "A4": "#ae587c"}
SCENARIO_LABELS = {
    "stable": "Estable",
    "covariate": "Deriva de covariables",
    "concept": "Deriva de concepto",
    "combined": "Deriva combinada",
}


def render_all_from_csv(root):
    """Regenerate all eight figure families from CSV alone, without model/data access."""
    root = Path(root)
    required = [
        "training_history.csv",
        "daily_metrics.csv",
        "actions.csv",
        "oracle_audit.csv",
        "reward_configs.csv",
        "sweep_summary.csv",
        "drift_samples.csv",
    ]
    missing = [name for name in required if not (root / name).is_file()]
    if missing:
        raise FileNotFoundError(f"CSV-only plotting missing: {missing}")
    history = pd.read_csv(root / "training_history.csv")
    daily = pd.read_csv(root / "daily_metrics.csv")
    actions = pd.read_csv(root / "actions.csv")
    audits = pd.read_csv(root / "oracle_audit.csv")
    configs = pd.read_csv(root / "reward_configs.csv")
    samples = pd.read_csv(root / "drift_samples.csv")
    manifest, smooth_rows, scenario_rows, action_means = [], [], [], []
    figures = root / "figures"
    figures.mkdir(exist_ok=True)

    def save(fig, directory, name, family, config_id="shared"):
        directory.mkdir(parents=True, exist_ok=True)
        fig.tight_layout()
        path = directory / f"{name}.png"
        fig.savefig(path, dpi=140, bbox_inches="tight")
        plt.close(fig)
        manifest.append(
            dict(
                family=family,
                config_id=config_id,
                path=path.relative_to(root).as_posix(),
            )
        )

    def markers(ax, scenario):
        if scenario != "stable":
            ax.axvline(ONSET, color="#777777", ls="--", lw=0.8)
        if scenario == "combined":
            ax.axvline(ONSET + CONCEPT_DELAY, color="#b57e35", ls=":", lw=0.8)

    def panels(title):
        fig, axes = plt.subplots(2, 2, figsize=(13, 8))
        fig.suptitle(title, fontsize=13)
        return fig, axes.flat

    # 1: sample-based plots retain enough raw values to avoid reconstructing a replay.
    if len(samples):
        hist_rows, relation_rows = [], []
        for scenario, filename in [
            ("covariate", "10_distribucion_vitales_covariate"),
            ("concept", "11_concept_drift_relacion"),
        ]:
            data = samples.loc[samples.scenario.eq(scenario)]
            first, last = int(data.day.min()), int(data.day.max())
            if scenario == "covariate":
                fig, axes = plt.subplots(2, 4, figsize=(16, 8))
            else:
                fig, axes = plt.subplots(2, 7, figsize=(19, 8))
            for j, vital in enumerate(VITALS):
                a = data.loc[data.day.eq(first) & data.vital.eq(vital)]
                b = data.loc[data.day.eq(last) & data.vital.eq(vital)]
                va = a.value.dropna().to_numpy()
                vb = b.value.dropna().to_numpy()
                ax = axes.flat[j] if scenario == "covariate" else axes[0, j]
                if not len(va) or not len(vb):
                    ax.set_visible(False)
                    continue
                lo, hi = np.nanpercentile(np.r_[va, vb], [0.5, 99.5])
                if hi <= lo:
                    hi = lo + 1e-6
                bins = np.linspace(lo, hi, 35)
                for day, values, color in [
                    (first, va, "#327ea4"),
                    (last, vb, "#ca9145"),
                ]:
                    density, _ = np.histogram(values, bins=bins, density=True)
                    for k, value in enumerate(density):
                        hist_rows.append(
                            dict(
                                scenario=scenario,
                                vital=vital,
                                day=day,
                                bin_left=bins[k],
                                bin_right=bins[k + 1],
                                density=value,
                            )
                        )
                    ax.hist(
                        values,
                        bins=bins,
                        density=True,
                        histtype="step",
                        lw=1.5,
                        color=color,
                        label=f"Día {day}",
                    )
                    ax.axvline(np.median(values), color=color, ls="--", lw=0.8)
                ax.set_title(
                    f"{vital} · KS={ks_2samp(va, vb, method='asymp').statistic:.2f}",
                    fontsize=10,
                )
                ax.tick_params(labelsize=8)
                if scenario == "concept":
                    edges = np.unique(np.nanquantile(va, np.linspace(0, 1, 11)))
                    if len(edges) < 2:
                        edges = np.array([va[0] - 1e-6, va[0] + 1e-6])
                    edges[0], edges[-1] = -np.inf, np.inf
                    for day, rows, color in [
                        (first, a, "#327ea4"),
                        (last, b, "#ca9145"),
                    ]:
                        clean = rows.loc[rows.value.notna()]
                        idx = np.clip(
                            np.searchsorted(edges, clean.value, side="right") - 1,
                            0,
                            len(edges) - 2,
                        )
                        rates = []
                        for k in range(len(edges) - 1):
                            labels = clean.loc[idx == k, "scenario_label"]
                            rate = float(labels.mean()) if len(labels) else np.nan
                            rates.append(rate)
                            relation_rows.append(
                                dict(
                                    vital=vital,
                                    day=day,
                                    decile=k + 1,
                                    edge_left=edges[k],
                                    edge_right=edges[k + 1],
                                    positive_rate=rate,
                                    n=len(labels),
                                )
                            )
                        axes[1, j].plot(
                            np.arange(1, len(rates) + 1),
                            rates,
                            "-o",
                            ms=3,
                            color=color,
                            label=f"Día {day}",
                        )
                    axes[1, j].set(xlabel="Decil fijo del día 1", ylim=(-0.02, 1.02))
                    axes[1, j].tick_params(labelsize=8)
            if scenario == "covariate":
                axes.flat[-1].axis("off")
            handles, labels = axes.flat[0].get_legend_handles_labels()
            fig.legend(
                handles, labels, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02)
            )
            fig.suptitle(
                f"{SCENARIO_LABELS[scenario]} · días {first} y {last}\n"
                "Histogramas: percentiles compartidos 0.5–99.5; etiquetas experimentales",
                fontsize=13,
            )
            save(fig, figures, filename, 1)
        pd.DataFrame(hist_rows).to_csv(root / "drift_histograms.csv", index=False)
        pd.DataFrame(relation_rows).to_csv(
            root / "concept_decile_relationship.csv", index=False
        )

    # 2: common return is comparable across objectives; native return remains visible.
    fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
    for i, cfg in configs.iterrows():
        h = history.loc[history.config_id.eq(cfg.config_id)].sort_values("episode")
        color = plt.get_cmap("tab10")(i)
        label = f"α={cfg.alpha:g}, λ={cfg.cost_lambda:g}"
        for ax, key in zip(axes, ["train_return", "train_common_return"]):
            rolling = h[key].rolling(100, min_periods=100)
            mean, std, var = rolling.mean(), rolling.std(ddof=1), rolling.var(ddof=1)
            for ep, raw, m, s, v in zip(h.episode, h[key], mean, std, var):
                smooth_rows.append(
                    dict(
                        config_id=cfg.config_id,
                        episode=ep,
                        metric=key,
                        window=100,
                        raw=raw,
                        mean=m,
                        std=s,
                        variance=v,
                    )
                )
            if mean.notna().any():
                ax.plot(h.episode, mean, color=color, label=label, lw=1.5)
                ax.fill_between(
                    h.episode, mean - std, mean + std, color=color, alpha=0.08
                )
            else:
                ax.plot(
                    h.episode,
                    h[key],
                    color=color,
                    label=label + " · crudo (smoke)",
                    lw=1,
                )
    axes[0].set(
        title="Retorno nativo: los pesos cambian su significado",
        ylabel="Retorno por episodio",
    )
    axes[1].set(
        title="Utilidad común: 0.5 AUC + 0.5 AP − 0.025 C",
        ylabel="Retorno común",
        xlabel="Episodio",
    )
    axes[0].legend(ncol=3, fontsize=8, loc="upper left")
    fig.suptitle("DDQN v6 · media móvil de 100 episodios y ±1 desviación estándar")
    save(fig, figures, "01_comparacion_rewards", 2)

    for cfg in configs.itertuples(index=False):
        cid = cfg.config_id
        dest = figures / cid
        h = history.loc[history.config_id.eq(cid)].sort_values("episode")
        d = daily.loc[daily.config_id.eq(cid)]
        ac = actions.loc[actions.config_id.eq(cid)]
        au = audits.loc[audits.config_id.eq(cid)]
        title = f"α={cfg.alpha:g}, λ={cfg.cost_lambda:g}"
        # 3: filtering the log does not train four separate agents.
        fig, axes = panels(f"Retorno por escenario · {title}")
        for ax, scenario in zip(axes, SCENARIOS):
            sh = h.loc[h.scenario.eq(scenario)]
            rolling = sh.train_return.rolling(15, min_periods=15)
            mean, std, var = rolling.mean(), rolling.std(ddof=1), rolling.var(ddof=1)
            for ep, raw, m, s, v in zip(sh.episode, sh.train_return, mean, std, var):
                scenario_rows.append(
                    dict(
                        config_id=cid,
                        scenario=scenario,
                        episode=ep,
                        window=15,
                        raw=raw,
                        mean=m,
                        std=s,
                        variance=v,
                    )
                )
            if mean.notna().any():
                ax.plot(
                    sh.episode,
                    mean,
                    color="#176ba5",
                    label="Media móvil: 15 apariciones",
                )
                ax.fill_between(
                    sh.episode,
                    mean - std,
                    mean + std,
                    color="#176ba5",
                    alpha=0.18,
                    label="±1 DE",
                )
            else:
                ax.plot(
                    sh.episode,
                    sh.train_return,
                    label="Crudo: menos de 15 apariciones",
                    lw=1,
                )
            ax.set(
                title=f"{SCENARIO_LABELS[scenario]} · {len(sh)} episodios",
                xlabel="Episodio global",
                ylabel="Retorno nativo",
            )
            ax.grid(alpha=0.2)
            ax.legend(fontsize=7)
        save(fig, dest, "02_retorno_por_escenario", 3, cid)
        # 5: totals are read from the CSV, not hardcoded to the 60 of old screenshots.
        fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
        for ax, phase, prefix in [
            (axes[0], "train", "train_count_"),
            (axes[1], "validation", "val_count_"),
        ]:
            sub = h if phase == "train" else h.loc[h.validation_return.notna()]
            bottom = np.zeros(len(sub))
            total = float(sub[[prefix + a for a in ACTIONS]].sum(axis=1).max())
            for a in ACTIONS:
                values = sub[prefix + a].to_numpy(dtype=float)
                ax.bar(
                    sub.episode,
                    values,
                    bottom=bottom,
                    width=1 if phase == "train" else 7,
                    color=ACTION_COLORS[a],
                    label=a,
                )
                bottom += values
                action_means.append(
                    dict(
                        config_id=cid,
                        phase=phase,
                        action=a,
                        mean_count=float(values.mean()),
                        mean_proportion=float(
                            values.sum()
                            / sub[[prefix + x for x in ACTIONS]].to_numpy().sum()
                        ),
                    )
                )
            ax.set(
                title=f"{phase}: {total:g} decisiones por barra",
                ylabel="Decisiones",
                ylim=(0, total * 1.03),
            )
            ax.legend(ncol=4, fontsize=8)
        axes[-1].set_xlabel("Episodio")
        fig.suptitle(f"Acciones durante el aprendizaje · {title}")
        save(fig, dest, "03_acciones_entrenamiento_validacion", 5, cid)
        means = pd.DataFrame(
            [r for r in action_means if r["config_id"] == cid and r["phase"] == "train"]
        )
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].bar(
            means.action,
            means.mean_count,
            color=[ACTION_COLORS[a] for a in means.action],
        )
        axes[1].bar(
            means.action,
            100 * means.mean_proportion,
            color=[ACTION_COLORS[a] for a in means.action],
        )
        axes[0].set(ylabel="Decisiones promedio por episodio")
        axes[1].set(ylabel="Porcentaje de decisiones", ylim=(0, 100))
        fig.suptitle(f"Uso promedio de acciones · {title}")
        save(fig, dest, "04_acciones_promedio", 5, cid)
        # 6 and 8: the same held-out B episode for every policy and configuration.
        for versus, name, family in [
            (False, "05_auc_politicas", 6),
            (True, "09_auc_dqn_vs_a1", 8),
        ]:
            fig, axes = panels(f"AUC diaria en B-test · {title}")
            policies = (
                ["A1", "dqn"] if versus else ["A1", "A2", "A3", "A4", "random", "dqn"]
            )
            for ax, scenario in zip(axes, SCENARIOS):
                for policy in policies:
                    g = d.loc[
                        d.scenario.eq(scenario) & d.policy.eq(policy)
                    ].sort_values("day")
                    color = ACTION_COLORS.get(
                        policy, "#176ba5" if policy == "dqn" else "#766b62"
                    )
                    ax.plot(
                        g.day,
                        g.auc,
                        label=policy,
                        color=color,
                        marker=".",
                        ms=3,
                        lw=2 if policy == "dqn" else 1.2,
                        ls="-" if policy == "dqn" else "--",
                    )
                markers(ax, scenario)
                ax.set(
                    title=SCENARIO_LABELS[scenario],
                    xlabel="Día simulado",
                    ylabel="ROC-AUC",
                    ylim=(0, 1),
                )
                ax.legend(ncol=3, fontsize=7)
            save(fig, dest, name, family, cid)
        # 7: actual action path and a myopic counterfactual plot, explicitly distinct.
        fig, axes = panels(f"Acciones DDQN en B-test · {title}")
        for ax, scenario in zip(axes, SCENARIOS):
            g = ac.loc[ac.scenario.eq(scenario) & ac.policy.eq("dqn")].sort_values(
                "day"
            )
            ax.step(g.day, g.executed + 1, where="mid", color="#666666", lw=0.8)
            for i, a in enumerate(ACTIONS):
                chosen = g.loc[g.executed.eq(i)]
                ax.scatter(
                    chosen.day,
                    chosen.executed + 1,
                    color=ACTION_COLORS[a],
                    label=a,
                    s=25,
                )
            markers(ax, scenario)
            ax.set(
                title=SCENARIO_LABELS[scenario],
                xlabel="Día",
                yticks=[1, 2, 3, 4],
                yticklabels=ACTIONS,
                ylim=(0.7, 4.3),
            )
        save(fig, dest, "06_acciones_episodio", 7, cid)
        fig, axes = panels(f"Oracle inmediato desde el mismo pipeline · {title}")
        for ax, scenario in zip(axes, SCENARIOS):
            sub = au.loc[au.scenario.eq(scenario)]
            for a in ACTIONS:
                g = sub.loc[sub.action.eq(a)].sort_values("day")
                ax.plot(
                    g.day,
                    g.common_reward,
                    label=a,
                    color=ACTION_COLORS[a],
                    lw=1.2,
                    marker=".",
                )
            chosen = sub.loc[sub.chosen.astype(str).str.lower().eq("true")]
            ax.scatter(
                chosen.day,
                chosen.common_reward,
                color="black",
                s=17,
                label="Elegida",
                zorder=4,
            )
            markers(ax, scenario)
            ax.set(
                title=SCENARIO_LABELS[scenario],
                xlabel="Día",
                ylabel="Utilidad inmediata común",
            )
            ax.legend(ncol=3, fontsize=7)
        save(fig, dest, "07_oracle_utilidad_comun", 7, cid)
        # 4: raw physical/statistical values and network inputs are separate figures.
        for scenario in SCENARIOS:
            g = d.loc[d.scenario.eq(scenario) & d.policy.eq("dqn")].sort_values("day")
            ga = ac.loc[ac.scenario.eq(scenario) & ac.policy.eq("dqn")].sort_values(
                "day"
            )
            fig, axes = plt.subplots(4, 2, figsize=(13, 11), sharex=True)
            fields = [
                ("ks", "KS"),
                ("psi", "PSI"),
                ("auc_drop", "Caída AUC"),
                ("average_precision", "AP"),
                ("auc", "AUC"),
                ("prevalence", "Prevalencia"),
                ("obs_valid", "Validez"),
            ]
            for ax, (key, label) in zip(axes.flat, fields):
                ax.plot(g.day, g[key], marker=".", lw=1.2)
                ax.set_ylabel(label)
                markers(ax, scenario)
                ax.grid(alpha=0.15)
            ax = axes.flat[-1]
            ax.step(ga.day, ga.executed + 1, where="mid")
            ax.set(yticks=[1, 2, 3, 4], yticklabels=ACTIONS, ylabel="Acción")
            for ax in axes[-1]:
                ax.set_xlabel("Día simulado")
            fig.suptitle(f"Monitoreo crudo · {SCENARIO_LABELS[scenario]} · {title}")
            save(fig, dest, f"08_monitoreo_crudo_{scenario}", 4, cid)
            fig, axes = plt.subplots(3, 2, figsize=(12, 8), sharex=True)
            for ax, key in zip(axes.flat, OBS_NAMES):
                ax.plot(g.day, g[f"obs_{key}"], marker=".", lw=1.2)
                ax.set(ylabel=f"Entrada: {key}", ylim=(-0.03, 1.03))
                markers(ax, scenario)
            axes.flat[-1].axis("off")
            fig.suptitle(
                f"Observaciones continuas recibidas por la red · {scenario} · {title}"
            )
            save(fig, dest, f"08_observaciones_normalizadas_{scenario}", 4, cid)
        fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
        axes[0].plot(h.episode, h.epsilon)
        axes[0].set_ylabel("Epsilon")
        axes[1].plot(h.episode, h.loss, lw=0.7)
        axes[1].set(ylabel="Huber media por episodio", xlabel="Episodio")
        fig.suptitle(f"Diagnóstico interno · {title}")
        save(fig, dest, "12_epsilon_huber", 9, cid)
    pd.DataFrame(smooth_rows).to_csv(root / "training_smoothing.csv", index=False)
    pd.DataFrame(scenario_rows).to_csv(root / "scenario_smoothing.csv", index=False)
    pd.DataFrame(action_means).to_csv(root / "action_averages.csv", index=False)
    manifest.extend(render_temporal_concept_from_csv(root))
    pd.DataFrame(manifest).to_csv(root / "figure_manifest.csv", index=False)
    print(f"[plots] {len(manifest)} PNG regenerated exclusively from CSV", flush=True)
    return pd.DataFrame(manifest)

## Orquestación, persistencia e integridad


In [ ]:
def rescore_frame(data, config):
    """Reuse measured baseline trajectories without rerunning timed interventions."""
    data = data.copy()
    data["config_id"], data["alpha"], data["cost_lambda"] = (
        config.config_id,
        config.alpha,
        config.cost_lambda,
    )
    valid = data.predictive_valid.astype(bool)
    data["auc_component"] = (1 - config.alpha) * data.auc.fillna(0) * valid
    data["ap_component"] = config.alpha * data.average_precision.fillna(0) * valid
    data["predictive_utility"] = data.auc_component + data.ap_component
    data["cost_penalty"] = config.cost_lambda * data.cost_relative
    data["reward"] = data.predictive_utility - data.cost_penalty
    if "warmup" in data:
        data.loc[data.warmup, "reward"] = np.nan
    return data


def export_drift_tables(schedules):
    """Export raw drift samples, patient decisions and small temporal plot tables."""
    samples, calendar, audits, timelines = [], [], [], []
    for scenario, schedule in schedules.items():
        audit = concept_audit_frame(schedule)
        if not audit.empty:
            audit["phase"], audit["replay_id"] = "test", "b_test"
            audits.append(audit)
            persist_concept_audit(schedule, "test", "b_test")
            categories = [audit.reason.eq("admitted_before_drift"),
                          audit.effective_shift_hours.lt(0), audit.effective_shift_hours.gt(0),
                          audit.converted, audit.reason.eq("negative_unchanged")]
            selected = []
            for condition in categories:
                subset = audit.loc[condition].sort_values(
                    ["event_censored_in_replay", "replay_rows", "source_set", "patient_id"],
                    ascending=[True, False, True, True])
                if not subset.empty:
                    row = subset.iloc[0]
                    selected.append((row.source_set, row.patient_id))
            if not selected:
                selected = list(audit[["source_set", "patient_id"]].head(5).itertuples(index=False, name=None))
            for site, pid in dict.fromkeys(selected):
                mask = schedule.source_set.astype(str).eq(str(site)) & schedule.patient_id.astype(str).eq(str(pid))
                rows = schedule.loc[mask, ["source_set", "patient_id", "ICULOS", "hour", "day",
                                           "scenario", "SepsisLabel", "scenario_label"]].copy()
                info = audit.loc[audit.source_set.eq(site) & audit.patient_id.eq(pid)].iloc[0]
                for key in ("reason", "admission_day", "concept_start_day", "original_first_positive",
                            "synthetic_first_positive", "effective_shift_hours", "event_censored_in_replay"):
                    rows[key] = info[key]
                timelines.append(rows)
        for day, g in schedule.groupby("day"):
            admitted = audit.loc[audit.admission_day.eq(day)] if not audit.empty else audit
            calendar.append(dict(
                injector_version=schedule.attrs["injector_version"], scenario=scenario, day=day,
                cov_level=float(g.cov_level.iloc[0]), concept_level=float(g.concept_level.iloc[0]),
                original_prevalence=float(g.SepsisLabel.mean()), scenario_prevalence=float(g.scenario_label.mean()),
                flip_rate=float(np.mean(g.scenario_label != g.SepsisLabel)),
                patients=int(g.patient_id.nunique()), rows=len(g),
                concept_start_day=int(audit.concept_start_day.iloc[0]) if not audit.empty else np.nan,
                new_admissions=len(admitted),
                eligible_admissions=int(admitted.eligible.sum()) if len(admitted) else 0,
                converted_admissions=int(admitted.converted.sum()) if len(admitted) else 0,
                positive_admissions=int(admitted.original_positive.sum()) if len(admitted) else 0,
            ))
        if scenario in ("covariate", "concept"):
            rows = schedule.loc[schedule.day.isin([1, int(schedule.day.max())])]
            samples.append(rows[["scenario", "day", "SepsisLabel", "scenario_label", *VITALS]].melt(
                id_vars=["scenario", "day", "SepsisLabel", "scenario_label"],
                value_vars=VITALS, var_name="vital", value_name="value"))
    table("drift_calendar", calendar)
    table("drift_samples", pd.concat(samples, ignore_index=True))
    if audits:
        table("concept_patients", pd.concat(audits, ignore_index=True))
    if timelines:
        table("concept_timelines", pd.concat(timelines, ignore_index=True))


def verify_temporal_labels(schedules):
    """Check generated labels and patient decisions without using any classifier."""
    checks = []
    for scenario, schedule in schedules.items():
        audit = concept_audit_frame(schedule)
        assert schedule.scenario_label.isin([0, 1]).all()
        if audit.empty:
            assert np.array_equal(schedule.SepsisLabel, schedule.scenario_label)
        else:
            assert audit.conversion_probability.between(0, .10).all()
            positive = audit.loc[audit.original_positive & audit.eligible]
            assert positive.synthetic_first_positive.notna().all()
            assert positive.effective_shift_hours.abs().le(positive.max_shift_hours).all()
            assert positive.synthetic_first_positive.between(positive.first_iculos, positive.last_iculos).all()
            by_patient = audit.set_index(["source_set", "patient_id"])
            for (site, pid), rows in schedule.groupby(["source_set", "patient_id"]):
                info = by_patient.loc[(str(site), str(pid))]
                if info.eligible:
                    ordered = rows.sort_values("ICULOS")
                    assert np.all(np.diff(ordered.scenario_label) >= 0)
                    expected = (ordered.ICULOS >= info.synthetic_first_positive).astype(np.int8)
                    np.testing.assert_array_equal(ordered.scenario_label, expected)
                else:
                    np.testing.assert_array_equal(rows.scenario_label, rows.SepsisLabel)
        checks.append(dict(check="temporal_patient_labels", artifact=scenario, passed=True, rows=len(schedule)))
    return checks


def verify_exports(frame, threshold):
    """Reconstruct predictive metrics from exported probabilities and check causality."""
    checks = []
    daily = pd.read_csv(OUT / "daily_metrics.csv")
    adaily = pd.read_csv(OUT / "a_daily_metrics.csv")
    original = frame.set_index(["patient_id", "ICULOS"]).SepsisLabel
    for path in sorted((OUT / "predictions").glob("*.csv")):
        rows = pd.read_csv(path)
        assert not rows.duplicated(["patient_id", "ICULOS"]).any()
        assert rows.groupby("hour").size().eq(BEDS).all()
        assert (
            rows.groupby("patient_id")
            .ICULOS.apply(lambda s: s.is_monotonic_increasing)
            .all()
        )
        assert (rows.label_available_day == rows.day + LABEL_DELAY).all()
        expected = original.reindex(
            pd.MultiIndex.from_frame(rows[["patient_id", "ICULOS"]])
        ).to_numpy()
        assert np.array_equal(expected, rows.SepsisLabel)
        if path.name.startswith("a_test"):
            ref = adaily.loc[adaily.partition.eq("a_test")].set_index("day")
            assert len(rows) == A_DAYS * 24 * BEDS
        else:
            first = rows.iloc[0]
            ref = daily.loc[
                daily.config_id.eq(first.config_id)
                & daily.scenario.eq(first.scenario)
                & daily.policy.eq(first.policy)
            ].set_index("day")
            assert len(rows) == DAYS * 24 * BEDS and len(ref) == DAYS
        for day, g in rows.groupby("day"):
            labels = (
                [("scenario_label", "")]
                if path.name.startswith("a_test")
                else [("scenario_label", ""), ("SepsisLabel", "_original")]
            )
            for label, suffix in labels:
                reconstructed = classification_metrics(
                    g[label], g.probability, threshold
                )
                for name in [
                    "auc",
                    "average_precision",
                    "pr_auc_trapz",
                    "f1_positive",
                    "f1_macro",
                    "precision",
                    "recall",
                    "balanced_accuracy",
                ]:
                    value, saved = reconstructed[name], ref.loc[day, name + suffix]
                    assert (pd.isna(value) and pd.isna(saved)) or abs(
                        value - saved
                    ) < 1e-9, (path.name, day, name, value, saved)
        checks.append(
            dict(
                check="prediction_stream_and_metrics",
                artifact=path.name,
                passed=True,
                rows=len(rows),
            )
        )
    for filename in ["train_actions", "validation_actions", "actions"]:
        # Read only the columns involved in causal validation, even for 243000 training actions.
        wanted = [
            "day",
            "state_end_day",
            "training_max_day",
            "training_label_available_max",
            "label_received_day",
            "executed",
            "reward",
            "common_reward",
            "cost_relative",
            "cost_penalty",
            "alpha",
            "cost_lambda",
            "auc",
            "average_precision",
            "predictive_valid",
            *[f"mask_{a}" for a in ACTIONS],
            *[f"state_{i}" for i in range(OBS_DIM)],
        ]
        for ac in pd.read_csv(OUT / f"{filename}.csv", usecols=wanted, chunksize=20000):
            assert (ac.state_end_day == ac.day - 1).all()
            assert (ac.training_max_day < ac.day).all()
            assert (ac.training_label_available_max <= ac.day).all()
            assert (ac.label_received_day == ac.day + LABEL_DELAY).all()
            assert np.isfinite(
                ac[[f"state_{i}" for i in range(OBS_DIM)]].to_numpy()
            ).all()
            valid = ac.predictive_valid.astype(bool)
            expected = (
                (1 - ac.alpha) * ac.auc.fillna(0)
                + ac.alpha * ac.average_precision.fillna(0)
            ) * valid - ac.cost_lambda * ac.cost_relative
            common = (
                0.5 * ac.auc.fillna(0) + 0.5 * ac.average_precision.fillna(0)
            ) * valid - 0.025 * ac.cost_relative
            assert np.allclose(ac.reward, expected, atol=1e-12, rtol=0)
            assert np.allclose(ac.common_reward, common, atol=1e-12, rtol=0)
            for i, a in enumerate(ACTIONS):
                assert ac.loc[ac.executed.eq(i), f"mask_{a}"].all()
        checks.append(
            dict(
                check="causal_state_masks_reward",
                artifact=filename,
                passed=True,
                rows=0,
            )
        )
    history = pd.read_csv(OUT / "training_history.csv")
    assert len(history) == len(CONFIGS) * MAX_EPISODES
    assert history.groupby("config_id").size().eq(MAX_EPISODES).all()
    assert (
        history[[f"train_count_{a}" for a in ACTIONS]]
        .sum(axis=1)
        .eq(DAYS - WARMUP)
        .all()
    )
    valid = history.loc[history.validation_return.notna()]
    assert (
        valid[[f"val_count_{a}" for a in ACTIONS]]
        .sum(axis=1)
        .eq(4 * (DAYS - WARMUP))
        .all()
    )
    assert daily.loc[daily.warmup, "action"].eq("A1").all()
    assert (
        daily.groupby(["config_id", "scenario", "policy"])
        .label_received_day.max()
        .eq(DAYS + LABEL_DELAY)
        .all()
    )
    oracle = pd.read_csv(OUT / "oracle_audit.csv")
    assert oracle.regret.ge(-1e-12).all() and oracle.native_regret.ge(-1e-12).all()
    freeze = pd.read_csv(OUT / "controller_freeze.csv")
    assert freeze.passed.all() and freeze.oracle_training_transitions.eq(0).all()
    checks.extend(
        [
            dict(check=name, artifact="v6", passed=True, rows=len(history))
            for name in [
                "complete_grid_episodes",
                "27_decisions_full_or_5_smoke",
                "validation_four_scenarios",
                "warmup_and_final_label_release",
                "frozen_controllers",
                "myopic_oracle_outside_learning",
            ]
        ]
    )
    return table("integrity_checks", checks)


def run_experiment():
    """Prepare once, train nine independent agents, freeze selection, then evaluate test."""
    global RUN_OWNED
    if list(OUT.glob("*.csv")) or list((OUT / "predictions").glob("*.csv")):
        raise FileExistsError(
            "Output directory already contains a run; use a fresh DARL_DUMMY_OUTPUT"
        )
    RUN_OWNED = True
    reset_seed()
    checkpoint(
        "setup",
        requested_configs=len(CONFIGS),
        completed_configs=0,
        common_alpha=COMMON_CONFIG.alpha,
        common_lambda=COMMON_CONFIG.cost_lambda,
        early_stopping=False,
    )
    table(
        "reward_configs",
        [
            dict(
                config_id=c.config_id,
                alpha=c.alpha,
                cost_lambda=c.cost_lambda,
                reference_alpha=COMMON_CONFIG.alpha,
                reference_lambda=COMMON_CONFIG.cost_lambda,
                seed=SEED,
                episodes=MAX_EPISODES,
            )
            for c in CONFIGS
        ],
    )
    table(
        "normalization",
        [
            dict(
                feature="KS",
                formula="KS",
                range="[0,1]",
                fit_partition="none",
                reason="bounded distance",
            ),
            dict(
                feature="PSI",
                formula="PSI/(1+PSI)",
                range="[0,1)",
                fit_partition="none",
                reason="monotonic compression without clipping",
            ),
            dict(
                feature="DeltaAUC",
                formula="(AUC_reference-AUC+1)/2",
                range="[0,1]",
                fit_partition="none",
                reason="analytic bounded affine map",
            ),
            dict(
                feature="AP",
                formula="AP",
                range="[0,1]",
                fit_partition="none",
                reason="naturally bounded; retain original resolution",
            ),
            dict(
                feature="valid",
                formula="both_classes_and_finite_AUC_AP",
                range="{0,1}",
                fit_partition="none",
                reason="explicit imputation indicator",
            ),
        ],
    )
    table("concept_config", [asdict(TEMPORAL_CONCEPT_CONFIG)])
    frame, partitions, first_positive = load_frame()
    patient_metadata = build_patient_metadata(frame, partitions)
    table("patient_history_metadata", [
        {**{k: v for k, v in info.items() if k != "hours"}, "history_rows": len(info["hours"])}
        for info in patient_metadata.values()])
    a_replays = {
        name: make_replay(frame, partitions[name], A_DAYS, BEDS)
        for name in ("a_train", "a_validation", "a_test")
    }
    b_replays = {
        name: make_replay(frame, partitions[name])
        for name in ("b_train", "b_validation", "b_test")
    }
    rng = np.random.default_rng(SEED)
    train_bases = [make_replay(frame, partitions["b_train"], rng=rng) for _ in range(4)]
    table(
        "patient_partitions",
        [
            dict(partition=name, patient_id=p, ever_sepsis=p in first_positive)
            for name, ids in partitions.items()
            for p in ids
        ],
    )
    counts = []
    for name, replay in (
        a_replays
        | b_replays
        | {f"b_train_replay_{i}": r for i, r in enumerate(train_bases)}
    ).items():
        source = "b_train" if name.startswith("b_train_replay") else name
        counts.append(
            dict(
                partition=name,
                patients_available=len(partitions[source]),
                patients_used=replay.patient_id.nunique(),
                rows_consumed=len(replay),
                days=int(replay.day.max()),
                beds=BEDS,
                capacity_passed=True,
            )
        )
    table("partition_counts", counts)
    for name, replay in a_replays.items():
        a_replays[name] = inject(
            replay, "stable", 0.0, 0.0, ONSET, np.zeros(7), np.ones(7)
        )
    initial, auc_reference, ap_reference, threshold = prepare_initial(a_replays)
    initial.model.save_model(OUT / "models" / "initial_xgboost.ubj")
    table(
        "initial_transform",
        [
            dict(vital=v, median=float(initial.med[i]), iqr=float(initial.iqr[i]))
            for i, v in enumerate(VITALS)
        ],
    )
    cov_rate, concept_rate, cost_unit, calibrated = calibrate(
        b_replays["b_train"], initial, auc_reference, patient_metadata
    )
    manifest = episode_manifest(MAX_EPISODES)
    table("episode_manifest", manifest)
    summaries = []
    for config in CONFIGS:
        summary = train_agent(
            train_bases,
            b_replays["b_validation"],
            initial,
            auc_reference,
            cost_unit,
            cov_rate,
            concept_rate,
            patient_metadata,
            ap_reference,
            threshold,
            config,
            manifest,
        )
        summaries.append(summary)
        table("sweep_summary", summaries)
        checkpoint(
            "config_complete",
            completed_configs=len(summaries),
            active_config=config.config_id,
        )
    ranked = pd.DataFrame(summaries).sort_values(
        [
            "selection_score",
            "selection_ap",
            "selection_auc",
            "selection_seconds",
            "config_id",
        ],
        ascending=[False, False, False, True, True],
        kind="stable",
    )
    selected = ranked.iloc[0].config_id
    ranked["selected"] = ranked.config_id.eq(selected)
    ranked["selection_rank"] = np.arange(1, len(ranked) + 1)
    table("sweep_summary", ranked)
    table(
        "selection_freeze",
        [
            dict(
                selected_config_id=selected,
                source="B-validation_only",
                before_b_test=True,
                common_alpha=0.5,
                common_lambda=0.025,
                selected_checkpoint=int(ranked.iloc[0].selected_episode),
            )
        ],
    )
    checkpoint(
        "selection_frozen",
        selected_config_id=selected,
        completed_configs=len(summaries),
    )
    schedules = {
        s: inject(
            b_replays["b_test"],
            s,
            cov_rate,
            concept_rate,
            ONSET,
            initial.med,
            initial.iqr,
            patient_metadata,
            TEMPORAL_CONCEPT_CONFIG,
        )
        for s in SCENARIOS
    }
    temporal_checks = verify_temporal_labels(schedules)
    export_drift_tables(schedules)
    all_results = []
    for scenario, schedule in schedules.items():
        # Baselines are run once, then rescored from the exact same timed outcomes.
        baselines = {}
        for policy in (*ACTIONS, "random", "heuristic"):
            result, actions, daily, _ = roll(
                schedule,
                policy,
                initial,
                auc_reference,
                cost_unit,
                export=True,
                ap_reference=ap_reference,
                reward_config=COMMON_CONFIG,
                threshold=threshold,
                phase="test",
            )
            baselines[policy] = (result, actions, daily)
            append_table("baseline_actions", actions)
            append_table("baseline_daily_metrics", daily)
        for config in CONFIGS:
            agent = DQNAgent()
            weights = torch.load(
                OUT / "models" / f"{config.config_id}.pt",
                map_location=DEVICE,
                weights_only=True,
            )
            agent.online.load_state_dict(weights)
            agent.target.load_state_dict(weights)
            before = agent.fingerprint()
            result, actions, daily, audit = roll(
                schedule,
                "dqn",
                initial,
                auc_reference,
                cost_unit,
                agent,
                audit=True,
                export=True,
                ap_reference=ap_reference,
                reward_config=config,
                threshold=threshold,
                phase="test",
            )
            all_results.append(result)
            append_table("daily_metrics", daily)
            append_table("actions", actions)
            append_table("oracle_audit", audit)
            append_table(
                "controller_freeze",
                [
                    dict(
                        config_id=config.config_id,
                        alpha=config.alpha,
                        cost_lambda=config.cost_lambda,
                        phase="test",
                        episode=0,
                        weights_sha256=before,
                        passed=before == agent.fingerprint(),
                        oracle_training_transitions=0,
                    )
                ],
            )
            for policy, (base_result, base_actions, base_daily) in baselines.items():
                ba, bd = (
                    rescore_frame(base_actions, config),
                    rescore_frame(base_daily, config),
                )
                rescored = summarize_roll(bd, ba, threshold)
                rescored.update(
                    {k: v for k, v in base_result.items() if k.startswith("pooled_")}
                )
                all_results.append(rescored)
                append_table("actions", ba)
                append_table("daily_metrics", bd)
            table("policy_comparisons", all_results)
            checkpoint(
                "testing", active_config=config.config_id, completed_scenario=scenario
            )
    checks = verify_exports(frame, threshold)
    checks = table("integrity_checks", pd.concat([checks, pd.DataFrame(temporal_checks)], ignore_index=True))
    checkpoint("verified", integrity_passed=bool(checks.passed.all()))
    figures = render_all_from_csv(OUT)
    table(
        "artifact_manifest",
        [
            dict(path=p.relative_to(OUT).as_posix(), bytes=p.stat().st_size)
            for p in sorted(OUT.rglob("*"))
            if p.is_file()
        ],
    )
    checkpoint(
        "complete",
        status="complete",
        completed_configs=len(CONFIGS),
        total_episodes=len(CONFIGS) * MAX_EPISODES,
        selected_config_id=selected,
        integrity_passed=bool(checks.passed.all()),
        calibration_qualified=calibrated,
        auc_reference=auc_reference,
        ap_reference=ap_reference,
        threshold=threshold,
        cost_unit_s=cost_unit,
        cov_rate=cov_rate,
        concept_rate_level_per_day=concept_rate,
        prediction_files=len(list((OUT / "predictions").glob("*.csv"))),
        figure_files=len(figures),
        elapsed_s=time.perf_counter() - RUN_STARTED,
    )
    return ranked

## Ejecutar el modo seleccionado


In [ ]:
if __name__ == "__main__":
    if MODE == "plots":
        render_all_from_csv(OUT)
    else:
        try:
            run_experiment()
        except Exception as exc:
            if RUN_OWNED:
                checkpoint(
                    "failed",
                    status="failed",
                    error=repr(exc),
                    traceback=traceback.format_exc(),
                )
            print(traceback.format_exc(), flush=True)